# WetlandMapper, notebook 2: your own regions and wetlands

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manudeo/Wetlands_Indaba_2026_workshop/blob/main/notebooks/WetlandMapper_own_regions.ipynb)

**This is the next step after [notebook 1](https://colab.research.google.com/github/manudo/Wetlands_Indaba_2026_workshop/blob/main/notebooks/WetlandMapper_workshop.ipynb).** It is for people who have Google Earth Engine and have worked through notebook 1. There is no sample data here: everything is downloaded from Earth Engine for your own outlines, so the first run takes a few minutes.

It has the same three sections as notebook 1, with more choices:

| Section | What it does | Choices that are new here |
|---|---|---|
| **A. Find the wetlands** | Finds the wetlands inside a larger area and gives every wetland an **ID** | Satellite (Landsat, Sentinel-2, MODIS), water index, pixel size |
| **B. Wetlands through time** | For one wetland, a list of them, or all of them: wet area by year, trend, rainfall, wetness history, cover types by year | Same satellite choices; three rainfall datasets; reuse of Section A |
| **C. Cover types for a period** | Wetland Cover Types for any period you choose, at Level 1 and Level 2 | Start and end year and month; one composite, one per year or one per month |

**How the sections connect.** Run A on a larger area. Download its results when the notebook tells you to. Then choose your wetlands for B and C in one of three ways: from the wetlands Section A found (by ID, a list of IDs, or all of them), from the *patches file* that Section A saved earlier (so you can come back in a new session), or from a new outline file of a site you know (KMZ, KML, GeoPackage, GeoJSON or a zipped shapefile). If Section A already worked out the wetness history for your wetlands, Section B shows it and does not calculate it again.

**How to use it.** Run the cells in order. Cells with settings show a form on the right; click *Show code* to see what runs. Every section ends with a cell that saves its results (a zip you can download, or a copy in Google Drive). Everything is written to one folder, `outputs/<your project name>/`.

**A first look at the limits.** The images are read from Earth Engine with WetlandMapper's `fetch_xee`, which reads them lazily in chunks and suits large areas and long records. They are then held in memory (Colab has about 12 GB), and Earth Engine can still be slow or fail on very large requests. The notebook estimates the size before it downloads and stops when a request looks too large; for the heaviest options (one composite per month over a large area) it warns you and lets you try anyway. If `xee` is not installed it uses `fetch` instead, which asks for one whole image at a time and has a limit of about 40 MB per image.

Conveners of the workshop: Manudeo Singh and Stephen Tooth (Aberystwyth University) and Ruth-Mary Fisher (SANParks).

## Setup

Run this first. Enter your Earth Engine project ID and a name for your project. The notebook signs in to Earth Engine; in Colab a window may ask you to allow access. *DOWNLOAD_METHOD* chooses how images are read from Earth Engine: *xee* (default, for large areas) or *fetch* (simple, one whole image at a time).

In [ ]:
#@title 0. Setup: run this first (about a minute) { display-mode: "form" }
GEE_PROJECT = ""  #@param {type:"string"}
PROJECT_NAME = "my_project"  #@param {type:"string"}
DOWNLOAD_METHOD = "xee"  #@param ["xee", "fetch"]
# This notebook needs Google Earth Engine: enter your Google Cloud project ID.
# PROJECT_NAME names the folder that holds all your results (outputs/<PROJECT_NAME>/).
# DOWNLOAD_METHOD: xee (WetlandMapper's fetch_xee) reads the images lazily in chunks, which suits large areas and long records.
#   fetch asks Earth Engine for one whole image at a time (simple, but about 40 MB per image at most). xee is used when it is installed.

import os, sys, subprocess, importlib.util, json, re, zipfile, tempfile, shutil, datetime as dt
import xml.etree.ElementTree as ET
from pathlib import Path

REPO_URL = "https://github.com/manudeo/Wetlands_Indaba_2026_workshop.git"
WETLANDMAPPER_VERSION = "1.2.0"
try:
    import google.colab                      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_repo_root():
    # The folder that holds notebooks/ and data/boundaries. Works when the notebook is opened from notebooks/, from the
    # repository root, or from a folder that contains the repository.
    here = Path.cwd().resolve()
    for base in [here, *list(here.parents)[:5]]:
        if (base / "notebooks").is_dir() and (base / "data" / "boundaries").is_dir():
            return base
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "notebooks").is_dir() and (sub / "data" / "boundaries").is_dir():
            return sub
    return None

ROOT = find_repo_root()
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Open this notebook from inside the cloned repository (the folder that holds notebooks/ and data/).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)       # a fresh Colab session
    ROOT = Path.cwd() / "Wetlands_Indaba_2026_workshop"
os.chdir(ROOT)

def have(module):
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False

# Packages: pinned and installed in Colab; on your own computer they are checked, never changed.
MODULES = {"wetlandmapper": f"wetlandmapper[gee]=={WETLANDMAPPER_VERSION}", "geopandas": "geopandas", "rioxarray": "rioxarray", "netCDF4": "netCDF4", "scipy": "scipy", "ee": "earthengine-api"}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MODULES.values(), "xee", "dask[array]"], check=True)
else:
    missing = [pip for mod, pip in MODULES.items() if not have(mod)]
    if missing:
        raise ImportError("Missing packages: " + ", ".join(missing) + ".\nInstall them in your environment, for example:\n  pip install " + " ".join(f'"{m}"' for m in missing))

import warnings
import numpy as np, pandas as pd, xarray as xr, geopandas as gpd, rioxarray, shapely
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize, shapes
from scipy import ndimage
from scipy.stats import linregress, spearmanr
from shapely.geometry import box, shape as shp_shape

import wetlandmapper as wm
from wetlandmapper import (classify_dynamics, classify_wct_ema, classify_wct_ema_level2, compute_wet_frequency, build_run_manifest,
                           DYNAMICS_CLASSES, DYNAMICS_COLORS, WCT_CLASSES, WCT_COLORS, WCT_LEVEL2_CLASSES, WCT_LEVEL2_COLORS, summarize_dynamics)
from wetlandmapper.plotting import plot_dynamics, plot_wct, plot_wct_level2
from wetlandmapper import gee
from wetlandmapper.gee import fetch, fetch_xee
CUBE_METHOD = DOWNLOAD_METHOD
if CUBE_METHOD == "xee" and not (have("xee") and have("dask")):
    print("xee or dask is not installed, so 'fetch' is used instead (one request per image, about 40 MB per image at most). To use xee: pip install xee dask")
    CUBE_METHOD = "fetch"

warnings.filterwarnings("ignore", category=FutureWarning); warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11, "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False})
OUT = Path("outputs") / (re.sub(r"[^A-Za-z0-9_.-]+", "_", PROJECT_NAME) or "my_project"); OUT.mkdir(parents=True, exist_ok=True)

try:
    gee.init(project=GEE_PROJECT)                    # reuse saved credentials (your own computer, or an earlier sign-in)
except Exception:
    try:
        gee.authenticate(); gee.init(project=GEE_PROJECT)
    except Exception as e:
        raise RuntimeError(f"Earth Engine sign-in failed ({type(e).__name__}: {str(e)[:150]}). This notebook needs Earth Engine: check the project ID and that the Earth Engine API is enabled for it.") from e
print("Earth Engine ready.")

BOUND = ROOT / "data" / "boundaries"

# ================================================================ helpers copied from notebook 1
def read_kml_polygons(path):
    # Fallback KML/KMZ reader for when GDAL has no KML driver.
    p = Path(path)
    data = zipfile.ZipFile(p).read(next(n for n in zipfile.ZipFile(p).namelist() if n.lower().endswith(".kml"))) if p.suffix.lower() == ".kmz" else p.read_bytes()
    tag = lambda e: e.tag.split("}")[-1]
    ring = lambda b: [tuple(float(v) for v in t.split(",")[:2]) for t in next(c for c in b.iter() if tag(c) == "coordinates").text.split()]
    polys = []
    for poly in ET.fromstring(data).iter():
        if tag(poly) == "Polygon":
            outer, holes = None, []
            for b in poly:
                if tag(b) == "outerBoundaryIs": outer = ring(b)
                elif tag(b) == "innerBoundaryIs": holes.append(ring(b))
            if outer and len(outer) >= 4: polys.append(shapely.Polygon(outer, holes))
    return gpd.GeoDataFrame({"geometry": polys}, geometry="geometry", crs=4326)

def read_outline(path):
    # Polygons from .kmz, .kml, .gpkg, .geojson, a zipped shapefile or a .shp, as 2-D geometries in EPSG:4326.
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{p} not found.")
    try:
        g = gpd.read_file(f"zip://{p}") if p.suffix.lower() == ".zip" else gpd.read_file(p)
        ok = g.geom_type.isin(["Polygon", "MultiPolygon"]).any()
    except Exception:
        g, ok = None, False
    if not ok and p.suffix.lower() in (".kml", ".kmz"):
        g = read_kml_polygons(p)
    if g is None or g.crs is None:
        raise ValueError(f"{p.name} has no readable polygons or no coordinate system (a shapefile needs its .prj; or save as GeoPackage).")
    g = g.to_crs(4326)
    g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326)
    g = g[g.geom_type.isin(["Polygon", "MultiPolygon"])]
    if g.empty:
        raise ValueError(f"No polygons found in {p.name}: points and lines are not enough. Draw the area as a polygon (Google Earth: Add, Polygon) and save again.")
    return g.set_geometry([shapely.make_valid(x) for x in g.geometry], crs=4326).reset_index(drop=True)

def read_points(path):
    # Field points from a CSV with lat and lon columns, or from a kmz, kml, gpkg, shp or geojson of points.
    p = Path(path)
    if p.suffix.lower() in (".csv", ".txt"):
        df = pd.read_csv(p); low = {c.lower().strip(): c for c in df.columns}
        la = next((low[k] for k in ("lat", "latitude", "y") if k in low), None); lo = next((low[k] for k in ("lon", "lng", "long", "longitude", "x") if k in low), None)
        if la is None or lo is None:
            raise ValueError("The table needs latitude (lat) and longitude (lon) columns in decimal degrees.")
        g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lo], df[la]), crs=4326)
    else:
        g = gpd.read_file(p).to_crs(4326); g = g.set_geometry(shapely.force_2d(g.geometry.values), crs=4326); g = g[g.geom_type == "Point"]
    g = g.reset_index(drop=True); g.insert(0, "point_id", [f"P{i + 1}" for i in range(len(g))])
    return g

def upload_file():
    # Colab only: open the upload dialog and return the path of the uploaded file (a shapefile also needs its .dbf, .shx and .prj).
    try:
        from google.colab import files
    except ImportError as e:
        raise RuntimeError("Uploading works in Google Colab only. On your own computer choose 'path' and give the full path of the file.") from e
    up = files.upload(); tmp = Path(tempfile.mkdtemp(prefix="wm_upload_"))
    for n, b in up.items(): (tmp / Path(n).name).write_bytes(b)
    order = [".gpkg", ".kmz", ".kml", ".geojson", ".json", ".zip", ".shp", ".csv"]
    found = sorted((q for q in tmp.iterdir() if q.suffix.lower() in order), key=lambda q: order.index(q.suffix.lower()))
    if not found: raise ValueError("No readable file uploaded. Supported: " + ", ".join(order))
    return found[0]

def load_outline(source, path, example_path):
    # The outline chosen in a form: upload, a path, the example file, or none.
    if source == "none":
        return None
    return read_outline(upload_file() if source == "upload" else (path if source == "path in Drive or Colab" else example_path))

def geojson_of(gdf):
    return shapely.unary_union(gdf.geometry.values).__geo_interface__

def bounds_of(gdf, pad=0.004):
    w, s, e, n = gdf.total_bounds
    return (round(w - pad, 5), round(s - pad, 5), round(e + pad, 5), round(n + pad, 5))

def _years(obj):
    first = obj if isinstance(obj, xr.DataArray) else obj[list(obj.data_vars)[0]]
    return set(int(y) for y in pd.DatetimeIndex(first["time"].values).year)

def fin(da):
    return da.astype("float32").rio.write_crs("EPSG:4326")

def pixel_area_m2(da):
    dx, dy = (abs(float(r)) for r in da.rio.resolution())
    return dx * 111320 * np.cos(np.radians(float(da.y.mean()))) * dy * 110574

def inside_mask(gdf, like):
    ref = like.isel(time=0, drop=True) if "time" in like.dims else like
    def r(t): return rasterize([(g, 1) for g in gdf.geometry], out_shape=ref.shape, transform=ref.rio.transform(), fill=0, dtype="uint8", all_touched=t).astype(bool)
    m = r(False)
    return xr.DataArray(m if m.any() else r(True), dims=ref.dims, coords=ref.coords)

def utm_epsg(da):
    return (32700 if float(da.y.mean()) < 0 else 32600) + int((float(da.x.mean()) + 180) // 6) + 1

def write_geotiff(da, path, colors):
    # GeoTIFF in the local UTM zone (uint8, 255 = no data) with the class colours embedded, so QGIS shows them without a style file.
    r = max(round(abs(float(da.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(da.y.mean()))) / 5) * 5, 5)
    u = da.rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{utm_epsg(da)}", resolution=r, resampling=Resampling.nearest)
    arr = u.fillna(255).astype("uint8").values
    with rasterio.open(path, "w", driver="GTiff", height=arr.shape[0], width=arr.shape[1], count=1, dtype="uint8", crs=u.rio.crs, transform=u.rio.transform(), nodata=255, compress="deflate") as dst:
        dst.write(arr, 1)
        dst.write_colormap(1, {**{int(c): tuple(int(h[i:i + 2], 16) for i in (1, 3, 5)) + (255,) for c, h in colors.items()}, 255: (0, 0, 0, 0)})

def wetland_patches(dyn, wf, min_pixels=2):
    # Groups of touching wetland pixels (class above 0) as shapes in UTM, with area, main class and the share of years they were wet.
    epsg = utm_epsg(dyn); res = max(round(abs(float(dyn.rio.resolution()[0])) * 111320 * np.cos(np.radians(float(dyn.y.mean()))) / 5) * 5, 5)
    utm = xr.Dataset({"cls": dyn.astype("float32"), "wf": wf.astype("float32")}).rio.write_crs("EPSG:4326").rio.reproject(f"EPSG:{epsg}", resolution=res, resampling=Resampling.nearest)
    cls = utm["cls"].values; lab, n = ndimage.label(np.isfinite(cls) & (cls > 0), structure=np.ones((3, 3)))
    empty = gpd.GeoDataFrame({"patch_id": [], "area_ha": []}, geometry=[], crs=epsg)
    if n == 0: return empty
    sizes = ndimage.sum(lab > 0, lab, np.arange(1, n + 1)); keep = np.where(sizes >= min_pixels)[0] + 1
    remap = np.zeros(n + 1, dtype=np.int32); remap[keep] = np.arange(1, len(keep) + 1); lab = remap[lab]; ids = np.arange(1, len(keep) + 1)
    if len(ids) == 0: return empty
    dom = pd.DataFrame({"o": lab[lab > 0], "c": cls[lab > 0].astype(int)}).groupby("o")["c"].agg(lambda s: s.value_counts().idxmax())
    geoms = [(shp_shape(g), int(v)) for g, v in shapes(lab.astype("int32"), mask=lab > 0, transform=utm.rio.transform(), connectivity=8)]
    gdf = gpd.GeoDataFrame({"patch_id": [v for _, v in geoms]}, geometry=[g for g, _ in geoms], crs=epsg).dissolve(by="patch_id").reset_index()
    gdf["area_ha"] = (ndimage.sum(lab > 0, lab, gdf["patch_id"]) * res * res / 1e4).round(2)
    gdf["main_class"] = gdf["patch_id"].map(dom).map(DYNAMICS_CLASSES); gdf["percent_years_wet"] = ndimage.mean(utm["wf"].values, lab, gdf["patch_id"]).round(1)
    return gdf

def write_kmz(gdf, path):
    # Wetland patches for Google Earth or a phone.
    try:
        g = gdf[["patch_id", "area_ha", "main_class", "geometry"]].to_crs(4326); tmp = Path(tempfile.mkdtemp()) / "doc.kml"; g.to_file(tmp, driver="KML")
        with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z: z.write(tmp, "doc.kml")
    except Exception as e:
        print(f"  (KMZ not written: {type(e).__name__}; the GeoPackage has the same patches)")

def show(df):
    from IPython.display import display
    display(df)

MONTH_NAMES = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

def choose_months(mode, first_month, last_month, suggested=None):
    # Months from the drop-downs in a form. "all year" gives None. "suggested rainy season" gives the suggestion from the rainfall step.
    # "my own range of months" runs from the first month to the last month, both included; if the first month is later than the last, the range crosses New Year.
    if mode == "all year":
        return None
    if mode == "suggested rainy season":
        if not suggested:
            raise ValueError("The suggested rainy season comes from the rainfall step (Part B, step 1). Run it first, or choose your own range of months.")
        return list(suggested)
    a, b = MONTH_NAMES.index(first_month) + 1, MONTH_NAMES.index(last_month) + 1
    return list(range(a, b + 1)) if a <= b else list(range(a, 13)) + list(range(1, b + 1))

def check_dynamics_settings(n_years, years_per_period, wet_pct, persistent_pct):
    # The record is split into an early period (its first N years) and a recent period (its last N years).
    if 2 * years_per_period > n_years:
        raise ValueError(f"The record has {n_years} years, so each period can have at most {n_years // 2} years (you chose {years_per_period}). Lower 'years in each period'.")
    if persistent_pct < wet_pct:
        raise ValueError("'Persistent' needs a higher percentage than 'wet in at least': raise the persistent percentage or lower the other.")
    print(f"Settings: a pixel is wetland if it is wet in at least {wet_pct}% of years, and persistent if wet in at least {persistent_pct}%. "
          f"Early period = the first {years_per_period} years, recent period = the last {years_per_period} years.")

def class_table_for_display(df):
    # The class summary with plain column names.
    return df[["class", "pixel_count", "area_ha", "percent_of_valid"]].rename(columns={"class": "Class", "pixel_count": "Pixels", "area_ha": "Area (ha)", "percent_of_valid": "% of the area analysed"}).round({"Area (ha)": 1, "% of the area analysed": 1})

def suggest_months(rain, width=5):
    # The wettest run of months inside one calendar year (yearly composites are built by calendar year).
    clim = rain.assign(m=rain["date"].dt.month).groupby("m")["precip_mm"].mean().reindex(range(1, 13)).fillna(0)
    best = max(range(1, 13 - width + 1), key=lambda m0: clim.loc[m0:m0 + width - 1].sum())
    wrap = max(range(1, 13), key=lambda m0: sum(clim.loc[((m0 - 1 + k) % 12) + 1] for k in range(width)))
    return list(range(best, best + width)), [((wrap - 1 + k) % 12) + 1 for k in range(width)], clim

# ================================================================ sensors, pixel size and size checks
SENSORS = {
    "Landsat, all missions (1984 on)": dict(sensor="LandsatAll", scale=30, first=1984, revisit=8),
    "Landsat 5 (1984 to 2012)": dict(sensor="Landsat5", scale=30, first=1984, last=2012, revisit=16),
    "Landsat 7 (1999 on, stripes after 2003)": dict(sensor="Landsat7", scale=30, first=1999, revisit=16),
    "Landsat 8 (2013 on)": dict(sensor="Landsat8", scale=30, first=2013, revisit=16),
    "Landsat 9 (2021 on)": dict(sensor="Landsat9", scale=30, first=2021, revisit=16),
    "Sentinel-2 (2015 on)": dict(sensor="Sentinel2", scale=20, first=2015, revisit=5),
    "MODIS Terra (2000 on, 500 m)": dict(sensor="MODIS_Terra", scale=500, first=2000, revisit=8),
    "MODIS Aqua (2002 on, 500 m)": dict(sensor="MODIS_Aqua", scale=500, first=2002, revisit=8),
    "MODIS Terra and Aqua (2002 on, 500 m)": dict(sensor="MODISAll", scale=500, first=2002, revisit=4),
}

def sensor_settings(label, y0, y1, pixel_choice):
    # The Earth Engine sensor name, the pixel size in metres, and the years the sensor can cover.
    cfg = SENSORS[label]; scale = cfg["scale"] if pixel_choice == "automatic" else int(pixel_choice)
    lo, hi = cfg["first"], cfg.get("last", dt.date.today().year)
    y0n, y1n = max(int(y0), lo), min(int(y1), hi)
    if (y0n, y1n) != (int(y0), int(y1)):
        print(f"{label} covers {lo} to {hi}, so the years {y0n} to {y1n} are used.")
    if y0n > y1n:
        raise ValueError(f"{label} has no data for {y0} to {y1}. Choose other years or another sensor.")
    if cfg["sensor"].startswith("MODIS"):
        print("MODIS pixels are 500 m (25 ha): small wetlands will be missed or merged with their surroundings. It suits large lakes, lagoons and coasts.")
    return cfg["sensor"], scale, y0n, y1n

def grid_px(bounds, scale):
    step = scale / 111319.49
    return int((bounds[2] - bounds[0]) / step), int((bounds[3] - bounds[1]) / step)

def check_box(bounds, scale, what, limit=2.5e6):
    # With fetch, stop early when one image would be too large for one download. xee reads in chunks, so it has no such limit.
    nx, ny = grid_px(bounds, scale)
    if CUBE_METHOD == "fetch" and nx * ny > limit:
        raise ValueError(f"The {what} is too large for one download: {nx} x {ny} pixels at {scale} m (the limit is about {limit / 1e6:.1f} million). Use a smaller area or a larger pixel size.")
    return nx, ny

def check_workload(n_images, nx, ny, what, run_anyway, bands=3):
    # The images are assembled in memory (Colab has about 12 GB). Many images from a large area can fail or be very slow: warn, and stop unless the user chose to try anyway.
    values = n_images * nx * ny * bands
    if CUBE_METHOD == "fetch" and nx * ny * bands * 4 > 40e6:
        raise ValueError(f"One image of {what} would be {nx * ny * bands * 4 / 1e6:.0f} MB, above the 40 MB limit of one download with fetch. Use a smaller area or a larger pixel size, or use xee.")
    if values > (3e8 if CUBE_METHOD == "xee" else 6e7):
        msg = f"{what}: {n_images} images of {nx} x {ny} pixels (about {values / 1e6:.0f} million values, {values * 4 / 1e9:.1f} GB in memory). This may be slow or fail in Earth Engine."
        if not run_anyway:
            raise ValueError(msg + " Tick 'RUN_EVEN_IF_LARGE' to try anyway, or use a smaller area, a larger pixel size or a shorter period.")
        print("Warning: " + msg + " Trying anyway.")

# ================================================================ downloads with WetlandMapper's fetch
def load_cube(**kw):
    # Images from Earth Engine as one xarray (time, y, x). xee: WetlandMapper's fetch_xee reads them lazily in chunks and they are loaded here.
    # Time steps with no data at all are dropped, as with fetch. fetch: one request per image.
    if CUBE_METHOD != "xee":
        return fetch(**kw)
    cube = fetch_xee(**kw).rename({"lat": "y", "lon": "x"}).transpose("time", "y", "x").compute()
    first = cube if isinstance(cube, xr.DataArray) else cube[list(cube.data_vars)[0]]
    has_data = first.notnull().any(("y", "x")).values
    return cube.isel(time=np.where(has_data)[0]) if not has_data.all() else cube

def fetch_annual(geom, index, y0, y1, sensor, scale, cloud, **kw):
    # One composite per year. With fetch, years skipped by a failed download are fetched again once.
    base = dict(aoi=geom, sensor=sensor, scale=scale, temporal_aggregation="annual", use_slc_off=True, max_cloud_cover=float(cloud), project=GEE_PROJECT or None)
    out = load_cube(start=f"{y0}-01-01", end=f"{y1}-12-31", index=index, **base, **kw)
    for attempt in range(2 if CUBE_METHOD == "fetch" else 0):
        miss = sorted(set(range(y0, y1 + 1)) - _years(out))
        if not miss: break
        print(f"  fetching {len(miss)} missing year(s) again: {miss}")
        parts = []
        for y in miss:
            try: parts.append(load_cube(start=f"{y}-01-01", end=f"{y}-12-31", index=index, **base, **kw))
            except Exception as e: print(f"    {y}: {type(e).__name__}")
        parts = [q for q in parts if len(_years(q))]
        if parts: out = xr.concat([out] + parts, dim="time").sortby("time")
    return out

def water_images(geom, y0, y1, sensor, scale, cloud, index, months):
    # Yearly water image: the 90th percentile of the water index over the months chosen (water is often brief in dry places).
    kw = dict(months=list(months)) if months else {}
    da = fetch_annual(geom, index, y0, y1, sensor, scale, cloud, reduction_method="percentile", percentile=90.0, **kw)
    return fin(da[list(da.data_vars)[0]] if isinstance(da, xr.Dataset) else da)

def cover_indices(geom, y0, y1, sensor, scale, cloud, months):
    # MNDWI, NDVI and NDTI: the median of the months chosen, one image per year.
    ds = fetch_annual(geom, ["MNDWI", "NDVI", "NDTI"], y0, y1, sensor, scale, cloud, months=list(months), reduction_method="median")
    return xr.Dataset({v: fin(ds[v]) for v in ("MNDWI", "NDVI", "NDTI")})

def period_indices(geom, start, end, sensor, scale, cloud, mode, months=None):
    # MNDWI, NDVI and NDTI for a period: every scene ("one"), one composite per year ("year") or per month ("month").
    base = dict(aoi=geom, sensor=sensor, scale=scale, index=["MNDWI", "NDVI", "NDTI"], use_slc_off=True, max_cloud_cover=float(cloud), project=GEE_PROJECT or None)
    if mode == "one":
        ds = load_cube(start=start.isoformat(), end=(end + dt.timedelta(days=1)).isoformat(), temporal_aggregation="all", **base)
    elif mode == "year":
        kw = dict(months=list(months)) if months else {}
        ds = load_cube(start=f"{start.year}-01-01", end=f"{end.year}-12-31", temporal_aggregation="annual", reduction_method="median", **kw, **base)
    else:
        ds = load_cube(start=start.isoformat(), end=(end + dt.timedelta(days=1)).isoformat(), temporal_aggregation="monthly", reduction_method="median", **base)
        ds = ds.isel(time=np.where(pd.DatetimeIndex(ds.time.values) <= pd.Timestamp(end))[0])         # fetch adds the month of the end date plus one day: drop that empty extra month
    return xr.Dataset({v: fin(ds[v]) for v in ("MNDWI", "NDVI", "NDTI")})

# ================================================================ rainfall
RAIN_SOURCES = {
    "ERA5-Land, monthly (11 km, 1950 on)": dict(id="ECMWF/ERA5_LAND/MONTHLY_AGGR", band="total_precipitation_sum", factor=1000.0, scale=11132, kind="monthly"),
    "ERA5-Land, daily (11 km, 1950 on)": dict(id="ECMWF/ERA5_LAND/DAILY_AGGR", band="total_precipitation_sum", factor=1000.0, scale=11132, kind="daily"),
    "CHIRPS, daily (5 km, 1981 on, 50 S to 50 N)": dict(id="UCSB-CHG/CHIRPS/DAILY", band="precipitation", factor=1.0, scale=5566, kind="daily"),
}

def rain_series(bounds, y0, y1, source):
    # Monthly rainfall (mm) averaged over the box. A box smaller than one rainfall cell uses the cell at its centre.
    import ee
    cfg = RAIN_SOURCES[source]; w, s, e, n = bounds
    small = (e - w) < cfg["scale"] / 111319.49 and (n - s) < cfg["scale"] / 111319.49
    geom = ee.Geometry.Point([(w + e) / 2, (s + n) / 2]) if small else ee.Geometry.Rectangle([w, s, e, n])
    col = ee.ImageCollection(cfg["id"]).select(cfg["band"])
    def feature(date_string, img):
        d = img.reduceRegion(ee.Reducer.mean(), geom, cfg["scale"], maxPixels=int(1e9))
        return ee.Feature(None, {"date": date_string, "p": d.get(cfg["band"])})
    if cfg["kind"] == "monthly":
        col = col.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        feats = ee.FeatureCollection(col.map(lambda img: feature(img.date().format("YYYY-MM"), img))).getInfo()["features"]
    else:
        first = ee.Date.fromYMD(y0, 1, 1)
        def month(i):
            m0 = first.advance(ee.Number(i), "month")
            return feature(m0.format("YYYY-MM"), col.filterDate(m0, m0.advance(1, "month")).sum())
        feats = ee.FeatureCollection(ee.List.sequence(0, (y1 - y0 + 1) * 12 - 1).map(month)).getInfo()["features"]
    df = pd.DataFrame([f["properties"] for f in feats])
    if df.empty or df["p"].isna().all():
        raise ValueError(f"{source} has no data for this place and period (CHIRPS covers 50 S to 50 N; ERA5-Land covers the land surface).")
    df["precip_mm"] = pd.to_numeric(df["p"], errors="coerce") * cfg["factor"]; df["date"] = pd.to_datetime(df["date"] + "-01")
    return df.dropna(subset=["precip_mm"])[["date", "precip_mm"]].sort_values("date").reset_index(drop=True)

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = (np.radians(np.asarray(v, float)) for v in (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

def cluster_by_distance(lon, lat, max_km=500):
    # Group places so that each group is at most about 1000 km across (a group shares one rainfall series).
    lon, lat = np.asarray(lon, float), np.asarray(lat, float); lab = -np.ones(len(lon), int); k = 0
    while (lab < 0).any():
        i = int(np.where(lab < 0)[0][0]); d = haversine_km(lon[i], lat[i], lon, lat); lab[(lab < 0) & (d <= max_km)] = k; k += 1
    return lab

def rain_for_selection(sel, y0, y1, source):
    # One regional rainfall series for the selected wetlands, unless they are more than 1000 km apart: then one series per group.
    c = sel.geometry.representative_point(); lab = cluster_by_distance(c.x, c.y)
    sets = {}
    for k in sorted(set(lab)):
        sub = sel[lab == k]; sets[int(k)] = rain_series(bounds_of(sub, 0.0), y0, y1, source)
    return sets, pd.Series(lab, index=sel["wetland_id"].values)

# ================================================================ choosing wetlands
def parse_ids(text):
    # "12, 45, 301-310" -> ['12', '45', '301', ..., '310']
    out = []
    for part in re.split(r"[,\s;]+", str(text).strip()):
        if not part: continue
        m = re.fullmatch(r"(\d+)-(\d+)", part)
        out += [str(i) for i in range(min(int(m.group(1)), int(m.group(2))), max(int(m.group(1)), int(m.group(2))) + 1)] if m else [part]
    return out

def prepare_wetlands(gdf, id_col=""):
    # Wetlands as 2-D polygons in EPSG:4326 with a wetland_id, an area in hectares and an internal number _k.
    g = gdf.copy()
    if id_col:
        if id_col not in g.columns: raise ValueError(f"The file has no column '{id_col}'. Columns: {', '.join(map(str, g.columns[:12]))}")
        g["wetland_id"] = g[id_col]
    elif "wetland_id" in g.columns: pass
    elif "patch_id" in g.columns: g["wetland_id"] = g["patch_id"]
    else: g["wetland_id"] = np.arange(1, len(g) + 1)
    g = g.to_crs(4326); c = g.geometry.representative_point()
    epsg = (32700 if c.y.mean() < 0 else 32600) + int((c.x.mean() + 180) // 6) + 1
    g["area_ha"] = (g.to_crs(epsg).area / 1e4).round(2)
    g["wetland_id"] = g["wetland_id"].astype(int) if pd.api.types.is_integer_dtype(g["wetland_id"]) else g["wetland_id"].astype(str)
    if g["wetland_id"].duplicated().any(): raise ValueError("The wetland IDs are not unique. Choose another ID column, or leave it empty to use row numbers.")
    return g

def choose_wetlands(gdf, id_col, which, ids_text, min_ha, most):
    g = prepare_wetlands(gdf, id_col)
    if which == "only the IDs listed below":
        want = parse_ids(ids_text)
        if not want: raise ValueError("List the IDs you want, for example 12, 45, 301-310.")
        have_ids = g["wetland_id"].astype(str)
        missing = [w for w in want if w not in set(have_ids)]
        if missing: raise ValueError(f"These IDs are not in the file: {', '.join(missing[:20])}. IDs in the file run from {have_ids.iloc[0]} to {have_ids.iloc[-1]}.")
        g = g[have_ids.isin(want)]
    else:
        g = g[g["area_ha"] >= min_ha]
        if len(g) > most:
            print(f"There are {len(g)} wetlands of at least {min_ha:g} ha; the {most} largest are used. Raise MOST_WETLANDS to use more, or list the IDs you want.")
            g = g.sort_values("area_ha", ascending=False).head(most)
        if g.empty: raise ValueError(f"No wetland of at least {min_ha:g} ha. Lower SMALLEST_WETLAND_HA.")
    g = g.sort_values("wetland_id").reset_index(drop=True); g["_k"] = np.arange(1, len(g) + 1)
    return g

def wetland_labels(sel, like):
    # A raster of the same grid in which each pixel holds the number (_k) of the wetland it lies in, 0 outside.
    ref = like.isel(time=0, drop=True) if "time" in like.dims else like
    pairs = [(g, int(k)) for g, k in zip(sel.geometry, sel["_k"])]
    kw = dict(out_shape=ref.shape, transform=ref.rio.transform(), fill=0, dtype="int32")
    lab = rasterize(pairs, **kw)
    missing = [(g, k) for g, k in pairs if k not in set(np.unique(lab))]           # wetlands smaller than a pixel: use the touched pixels
    if missing: lab = np.where(lab == 0, rasterize(missing, all_touched=True, **kw), lab)
    return xr.DataArray(lab, dims=ref.dims, coords=ref.coords)

def class_counts(cls, labels, ks, codes):
    # Pixels of each class inside each wetland: a (wetlands x classes) array, and the number of valid pixels per wetland.
    a, l = np.asarray(cls), np.asarray(labels)
    table = np.stack([ndimage.sum((a == c).astype(float), l, ks) for c in codes], axis=1)
    return table, ndimage.sum(np.isfinite(a).astype(float), l, ks)

def region_box(sel, pad=0.0045):
    return box(*bounds_of(sel, pad))

def plot_outlines(ax, sel, max_labels=30):
    sel.boundary.plot(ax=ax, color="k", linewidth=0.9)
    if len(sel) <= max_labels:
        for r in sel.itertuples():
            p = r.geometry.representative_point(); ax.text(p.x, p.y, str(r.wetland_id), fontsize=7, ha="center", va="center", color="#c0392b", fontweight="bold")

def class_legend(fig, classes, colors, title):
    # One legend for a whole figure of class maps, placed to the right of the maps.
    from matplotlib.patches import Patch
    fig.legend([Patch(color=colors[c]) for c in classes], [classes[c] for c in classes], title=title, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8, title_fontsize=9)

def crop_to(da, geom, pad_px=1):
    # Crop a raster to the box of one polygon.
    w, s, e, n = geom.bounds; st = abs(float(da.rio.resolution()[0])) * pad_px
    return da.rio.clip_box(w - st, s - st, e + st, n + st)

# ================================================================ saving
def save_step(folder, mode, dest):
    # Zip the results of one step; in Colab download the zip or copy it to Drive; on your own computer copy it to a folder.
    src = OUT / folder
    if not src.exists() or not any(src.iterdir()):
        raise ValueError(f"There are no results in {src} yet: run the step first.")
    zip_path = Path(shutil.make_archive(str(OUT / f"{folder}_results"), "zip", src)); print("Zipped results:", zip_path)
    def copy_to(d):
        d = Path(d).expanduser(); d.mkdir(parents=True, exist_ok=True); shutil.copy(zip_path, d / zip_path.name)
        shutil.copytree(src, d / folder, dirs_exist_ok=True); print("Copied the folder and the zip to", d.resolve())
    if mode == "zip in the outputs folder":
        print("The files stay in", src.resolve(), "(on your own computer they are kept there; Colab deletes them when the session ends).")
    elif mode == "download zip (Colab)":
        if IN_COLAB:
            from google.colab import files
            files.download(str(zip_path))
        else:
            print("Not in Colab, so there is nothing to download: the zip is at", zip_path.resolve())
    elif mode == "copy to Google Drive (Colab)":
        if IN_COLAB:
            from google.colab import drive
            drive.mount("/content/drive"); copy_to(Path("/content/drive/MyDrive") / (dest or "WetlandMapper_results") / OUT.name)
        else:
            print("Not in Colab. Choose 'copy to a folder' and give a folder path, or use the zip at", zip_path.resolve())
    else:
        if not dest: raise ValueError("Give a folder path, for example ~/wetland_results")
        copy_to(dest)

def describe_outline(gdf, name, what):
    # Size and a small map of an outline.
    c = shapely.unary_union(gdf.geometry.values).centroid
    ha = gdf.to_crs((32700 if c.y < 0 else 32600) + int((c.x + 180) // 6) + 1).area.sum() / 1e4
    w, s_, e, n = gdf.total_bounds; nx, ny = grid_px((w, s_, e, n), 30)
    print(f"{what} '{name}': {len(gdf)} polygon(s), {ha:,.1f} ha, box {nx} x {ny} pixels at 30 m")
    fig, ax = plt.subplots(figsize=(4.5, 4)); gdf.plot(ax=ax, color="#76d7c4", edgecolor="k"); ax.set_title(name); ax.set_aspect(1 / np.cos(np.radians(float(c.y)))); plt.show()

print("WetlandMapper", wm.__version__, "|", "Colab" if IN_COLAB else "local", "| download method:", CUBE_METHOD, "| results go to", OUT)


## Your outlines

An outline is a **polygon**: KMZ or KML (drawn in Google Earth: Add, Polygon), GeoPackage, GeoJSON, a zipped shapefile, or a shapefile with its `.dbf`, `.shx` and `.prj`. Points and lines are not enough.

- **Section A** needs an **area to search**: a catchment, a reserve, a municipality, a farm. Load it in the cell below.
- **Sections B and C** need **wetlands**. You choose them in the cell *Choose your wetlands*, from Section A's results or from a file of your own, so there is nothing to load now.

The example outline is a 7 km area around a pan on the Agulhas Plain (South Africa), only to test the route.

In [ ]:
#@title Your area to search (Section A) { display-mode: "form" }
REGION_NAME = "my_area"  #@param {type:"string"}
REGION_OUTLINE = "example"  #@param ["upload", "path in Drive or Colab", "example", "none"]
REGION_PATH = ""  #@param {type:"string"}
# upload: a button opens a file dialog (Colab). path: for example /content/drive/MyDrive/area.kmz, or a path on your computer. example: a 7 km area to test the route.

own_region = load_outline(REGION_OUTLINE, REGION_PATH, ROOT / "data" / "examples" / "pan_region_outline.kmz")
if own_region is not None: describe_outline(own_region, REGION_NAME, "Area to search")
else: print("No area loaded. You can still choose wetlands from a file in the cell 'Choose your wetlands'.")

## Section A. Find the wetlands in an area

**The question.** Which places in this area behave like wetland, and which wetlands are they? Each wetland found gets an **ID**, which you can use in Sections B and C.

**How it works.** WetlandMapper looks at one satellite image per year. In each image every pixel is wet or dry: it counts as wet when its water index is above zero. Each year's image keeps the wettest-looking cloud-free value of the pixel (the 90th percentile of that year's observations), because in dry places water often comes only briefly. A pixel that was wet in enough years counts as **wetland**, and its **wetness history** gives it a class. Touching wetland pixels are grouped into **wetland patches**, each with an ID, an area, a main class and the share of years it was wet.

**The choices**

- **Satellite.** *Landsat, all missions* gives the longest record at 30 m. *Sentinel-2* gives 10 to 20 m but only since 2015. *MODIS* has 500 m pixels and a record since 2000: it suits large lakes, lagoons and coasts, and misses small wetlands. A single Landsat mission is useful to avoid the stripes of Landsat 7 after 2003.
- **Water index.** *AWEIsh* (default) is built to separate water from shadow and dark surfaces. *AWEInsh* is the version for places with little shadow. *MNDWI* and *NDWI* are simpler. Each counts as water above zero.
- **Pixel size.** *automatic* is 30 m for Landsat, 20 m for Sentinel-2 and 500 m for MODIS. A larger pixel makes a large area possible.
- **Months.** *all year*, or a range of months of each year (both ends included). A range that crosses New Year (for example November to March) mixes two seasons, because the yearly images are built per calendar year.
- **Dynamics settings.** *Wet in at least this % of years* (default 25) makes a pixel wetland. *Persistent if wet in at least this % of years* (default 75). *Years per period* (default 3): the record is split into an early period (its first N years) and a recent period (its last N years), and the two are compared. The 2022 paper used 5. *Smallest patch* drops patches with fewer pixels.

In [ ]:
#@title Section A. Find the wetlands { display-mode: "form" }
SENSOR_A = "Landsat, all missions (1984 on)"  #@param ["Landsat, all missions (1984 on)", "Landsat 5 (1984 to 2012)", "Landsat 7 (1999 on, stripes after 2003)", "Landsat 8 (2013 on)", "Landsat 9 (2021 on)", "Sentinel-2 (2015 on)", "MODIS Terra (2000 on, 500 m)", "MODIS Aqua (2002 on, 500 m)", "MODIS Terra and Aqua (2002 on, 500 m)"]
WATER_INDEX_A = "AWEIsh"  #@param ["AWEIsh", "AWEInsh", "MNDWI", "NDWI"]
FIRST_YEAR_A = 1990  #@param {type:"integer"}
LAST_YEAR_A = 2025  #@param {type:"integer"}
MONTHS_A = "all year"  #@param ["all year", "my own range of months"]
FIRST_MONTH_A = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
LAST_MONTH_A = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
PIXEL_SIZE_A = "automatic"  #@param ["automatic", "10", "20", "30", "60", "90", "250", "500", "1000"]
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
PERSISTENT_YEARS_PERCENT = 75  #@param {type:"slider", min:50, max:100, step:5}
YEARS_PER_PERIOD = 3  #@param {type:"slider", min:1, max:10, step:1}
SMALLEST_PATCH_PIXELS = 2  #@param {type:"slider", min:1, max:10, step:1}
MAX_SCENE_CLOUD_PERCENT_A = 40  #@param {type:"slider", min:10, max:80, step:5}
RUN_EVEN_IF_LARGE = False  #@param {type:"boolean"}
# RUN_EVEN_IF_LARGE: tick to try a request the notebook thinks is too large (it may be slow or fail).
# SENSOR_A and PIXEL_SIZE_A: see the text above. Years the sensor does not cover are dropped, with a note.
# MONTHS_A: "all year", or "my own range of months" with the first and last month (both included).
# MINIMUM_WET_YEARS_PERCENT: a pixel counts as wetland if it was wet in at least this % of the years.
# PERSISTENT_YEARS_PERCENT: a pixel is 'Persistent' if it was wet in at least this % of the years.
# YEARS_PER_PERIOD: the early period is the first N years and the recent period the last N years (the 2022 paper used 5).
# SMALLEST_PATCH_PIXELS: patches with fewer pixels are dropped. MAX_SCENE_CLOUD_PERCENT_A: scenes cloudier than this are not used.

assert own_region is not None, "Run the cell 'Your area to search' first and load an outline."
sensor_a, scale_a, y0a, y1a = sensor_settings(SENSOR_A, FIRST_YEAR_A, LAST_YEAR_A, PIXEL_SIZE_A)
months_a = choose_months(MONTHS_A, FIRST_MONTH_A, LAST_MONTH_A)
if months_a and months_a != list(range(min(months_a), max(months_a) + 1)):
    print("Note: the months cross New Year. Each yearly image takes them from one calendar year, so it mixes the end of one season with the start of the next.")
nx_a, ny_a = check_box(bounds_of(own_region, 0), scale_a, "area to search")
check_workload(y1a - y0a + 1, nx_a, ny_a, "yearly water images", RUN_EVEN_IF_LARGE, bands=1)
print(f"Downloading {y1a - y0a + 1} yearly images of {WATER_INDEX_A} from {SENSOR_A}, pixel {scale_a} m, months {'all' if months_a is None else months_a} (a few minutes)...")
aw_a = water_images(geojson_of(own_region), y0a, y1a, sensor_a, scale_a, MAX_SCENE_CLOUD_PERCENT_A, WATER_INDEX_A, months_a)
check_dynamics_settings(aw_a.sizes["time"], YEARS_PER_PERIOD, MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT)

out_a = OUT / "A_find_wetlands"; out_a.mkdir(exist_ok=True); name_a = re.sub(r"[^A-Za-z0-9_.-]+", "_", REGION_NAME)
mask_a = inside_mask(own_region, aw_a); ok_a = aw_a.notnull().any("time") & mask_a
dyn_a = classify_dynamics(aw_a, nYear=YEARS_PER_PERIOD, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=PERSISTENT_YEARS_PERCENT, nan_policy="valid").where(ok_a)
wf_a = compute_wet_frequency(aw_a, nan_policy="valid").where(ok_a)
px_a = pixel_area_m2(aw_a)
classes_a = summarize_dynamics_table = None
classes_a = summarize_dynamics(dyn_a, pixel_area=px_a, area_unit="ha").to_dataframe().drop(columns=["class_name"]).join(pd.Series(DYNAMICS_CLASSES, name="class"))
patches_a = wetland_patches(dyn_a, wf_a, SMALLEST_PATCH_PIXELS)
patches_a["wetland_id"] = patches_a["patch_id"]
wet_ha = float(classes_a.loc[classes_a.index != 0, "area_ha"].sum()); tot_ha = float(ok_a.sum()) * px_a / 1e4
print(f"Wetland found: {wet_ha:,.0f} ha of the {tot_ha:,.0f} ha analysed ({100 * wet_ha / tot_ha:.1f}%), in {len(patches_a):,} patches of at least {SMALLEST_PATCH_PIXELS} pixels (1 pixel = {px_a / 1e4:.3f} ha).")
if len(patches_a): print(f"The median patch is {patches_a['area_ha'].median():.2f} ha and the largest is {patches_a['area_ha'].max():,.1f} ha. Open sea, large lakes and reservoirs also show as persistent wetland: check the largest patches on a map.")
yrs_a = sorted(set(pd.DatetimeIndex(aw_a.time.values).year)); gaps_a = [y for y in range(yrs_a[0], yrs_a[-1] + 1) if y not in yrs_a]
if gaps_a: print(f"No yearly image for {gaps_a}: those years are left out.")
show(class_table_for_display(classes_a))

fig, ax = plot_dynamics(dyn_a, title=f"{name_a}: wetness history {yrs_a[0]} to {yrs_a[-1]}", figsize=(11, 5.4))
ax.set_aspect(1 / np.cos(np.radians(float(dyn_a.y.mean())))); fig.savefig(out_a / f"{name_a}_wetness_history_map.png", dpi=150, bbox_inches="tight"); plt.show()
if len(patches_a):
    big = patches_a.sort_values("area_ha", ascending=False).head(10)
    print("The ten largest patches (their IDs work in Sections B and C):")
    show(big[["wetland_id", "area_ha", "main_class", "percent_years_wet"]].rename(columns={"wetland_id": "ID", "area_ha": "Area (ha)", "main_class": "Main class", "percent_years_wet": "% of years wet"}).set_index("ID"))

write_geotiff(dyn_a, out_a / f"{name_a}_wetness_history.tif", DYNAMICS_COLORS)
class_table_for_display(classes_a).to_csv(out_a / f"{name_a}_wetness_history_classes.csv")
if len(patches_a):
    patches_a.to_file(out_a / f"{name_a}_wetland_patches.gpkg", layer="wetland_patches", driver="GPKG"); write_kmz(patches_a, out_a / f"{name_a}_wetland_patches.kmz")
    c4326 = patches_a.to_crs(4326).geometry.representative_point()
    patches_a.drop(columns="geometry").assign(lon=c4326.x.round(5).values, lat=c4326.y.round(5).values).to_csv(out_a / f"{name_a}_wetland_patches.csv", index=False)
SETTINGS_A = {"sensor": SENSOR_A, "water_index": WATER_INDEX_A, "pixel_m": scale_a, "years": [y0a, y1a], "months": "all" if months_a is None else months_a, "nYear": YEARS_PER_PERIOD,
              "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": PERSISTENT_YEARS_PERCENT, "smallest_patch_pixels": SMALLEST_PATCH_PIXELS, "max_cloud_percent": MAX_SCENE_CLOUD_PERCENT_A}
build_run_manifest(parameters={"section": "A", "area": name_a, **SETTINGS_A}, output_path=out_a / f"{name_a}_run_manifest.json")
STATE_A = dict(aw=aw_a, dyn=dyn_a, wf=wf_a, patches=patches_a, px_a=px_a, sensor=sensor_a, scale=scale_a, index=WATER_INDEX_A, months=months_a, y0=y0a, y1=y1a,
               nYear=YEARS_PER_PERIOD, thr_wet=MINIMUM_WET_YEARS_PERCENT, thr_persis=PERSISTENT_YEARS_PERCENT, bounds=tuple(aw_a.rio.bounds()), name=name_a)
print("\nFiles are in", out_a)
print("NEXT: save these results now (the next cell). Keep the file *_wetland_patches.gpkg: it holds the wetland IDs. In Sections B and C you can choose wetlands by ID from it, even in a new session, or upload a new outline of a site you know.")

**How to read it**

- **The sentences at the top** say how much wetland the satellite record shows and in how many patches.
- **The map** colours each wetland pixel by its wetness history. *Persistent*: wet in nearly every year (the stable core). *Intermittent*: wet in some years and dry in others, with no clear direction (it swings). *Intensifying* and *Diminishing*: wetter or drier in the recent period than in the early period. *New*: dry throughout the early period but wet in the recent one. *Lost*: wet in the early period but dry throughout the recent one.
- **The table of the ten largest patches** gives the IDs. Use them in Sections B and C.
- **Open sea, large lakes and reservoirs show as persistent wetland.** If the largest patch is one of those, leave it out when you choose wetlands (for example by listing the IDs you want).
- **Small, vegetated or briefly wet features** are seen poorly, most of all with large pixels.

Files are in `outputs/<project>/A_find_wetlands/`: a GeoTIFF with the class colours inside, the wetland patches as GeoPackage (keeps the IDs; QGIS and ArcGIS), KMZ (Google Earth) and CSV, the class table, the map and a record of your settings.

In [ ]:
#@title Save the results of Section A { display-mode: "form" }
SAVE_TO = "zip in the outputs folder"  #@param ["zip in the outputs folder", "download zip (Colab)", "copy to Google Drive (Colab)", "copy to a folder"]
DESTINATION = ""  #@param {type:"string"}
# download zip (Colab): the browser downloads a zip of this section's folder.
# copy to Google Drive (Colab): DESTINATION is a folder name inside My Drive (default WetlandMapper_results).
# copy to a folder: DESTINATION is a folder path on your computer, for example ~/wetland_results.
save_step("A_find_wetlands", SAVE_TO, DESTINATION)

## Choose your wetlands (for Sections B and C)

Sections B and C run on **the wetlands you choose here**: one, several or all. Choose them in one of three ways:

- **wetlands found in Section A**: the patches Section A found in this session.
- **patches file from an earlier Section A run**: the `*_wetland_patches.gpkg` file you saved, so you can use the IDs in a new session.
- **an outline file**: your own outline of a site you know (KMZ, KML, GeoPackage, GeoJSON or zipped shapefile). One polygon makes a single wetland; many polygons make a list. Each polygon gets an ID from the column you name in *ID_COLUMN*, or its row number if you leave that empty.

Then choose **all** of them or **only the IDs listed**, such as `12, 45, 301-310`. With *all*, wetlands smaller than *SMALLEST_WETLAND_HA* are left out, and at most *MOST_WETLANDS* of the largest are used. A list of IDs is used as it is.

All the wetlands are downloaded together as one box, and each wetland is read from it, so many wetlands in one region cost no more downloads than one. If your wetlands are more than 1000 km apart they cannot share one rainfall series, and the notebook says so.

In [ ]:
#@title Choose your wetlands { display-mode: "form" }
WETLAND_SOURCE = "wetlands found in Section A"  #@param ["wetlands found in Section A", "patches file from an earlier Section A run", "an outline file"]
FILE_SOURCE = "upload"  #@param ["upload", "path in Drive or Colab"]
FILE_PATH = ""  #@param {type:"string"}
ID_COLUMN = ""  #@param {type:"string"}
WHICH_WETLANDS = "all"  #@param ["all", "only the IDs listed below"]
WETLAND_IDS = ""  #@param {type:"string"}
SMALLEST_WETLAND_HA = 1.0  #@param {type:"number"}
MOST_WETLANDS = 200  #@param {type:"integer"}
# WETLAND_SOURCE: where the wetlands come from. FILE_SOURCE and FILE_PATH are used only for the second and third options.
# ID_COLUMN: for an outline file, the column that holds each wetland's ID or name (empty = row numbers 1, 2, 3, ...).
# WHICH_WETLANDS: all, or only the IDs listed below. WETLAND_IDS: for example 12, 45, 301-310.
# SMALLEST_WETLAND_HA and MOST_WETLANDS apply to 'all' only.

if WETLAND_SOURCE == "wetlands found in Section A":
    assert "STATE_A" in globals() and len(STATE_A["patches"]), "Section A has not been run in this session (or found no wetland). Run it, or choose another source."
    source_gdf, source_name = STATE_A["patches"], "Section A"
else:
    f = upload_file() if FILE_SOURCE == "upload" else Path(FILE_PATH)
    if WETLAND_SOURCE.startswith("patches"):
        source_gdf = gpd.read_file(f, layer="wetland_patches") if str(f).lower().endswith(".gpkg") else gpd.read_file(f)
    else:
        source_gdf = read_outline(f)
    source_name = Path(f).name
SEL = choose_wetlands(source_gdf, ID_COLUMN if WETLAND_SOURCE == "an outline file" else "", WHICH_WETLANDS, WETLAND_IDS, SMALLEST_WETLAND_HA, MOST_WETLANDS)
SEL_SOURCE = WETLAND_SOURCE
cen = SEL.geometry.representative_point(); groups = cluster_by_distance(cen.x, cen.y)
print(f"{len(SEL)} wetland(s) chosen from {source_name}: " + (f"{SEL['area_ha'].iloc[0]:,.2f} ha." if len(SEL) == 1 else f"{SEL['area_ha'].sum():,.1f} ha in total, from {SEL['area_ha'].min():,.2f} to {SEL['area_ha'].max():,.2f} ha."))
if len(set(groups)) > 1:
    print(f"These wetlands fall in {len(set(groups))} groups that are more than 1000 km apart. Each group gets its own rainfall series, but they also need separate downloads, and one box around all of them would be far too large. Choose the wetlands of one region at a time.")
w_, s_, e_, n_ = SEL.total_bounds
print(f"Box around them: {haversine_km(w_, (s_ + n_) / 2, e_, (s_ + n_) / 2):,.1f} km wide and {haversine_km((w_ + e_) / 2, s_, (w_ + e_) / 2, n_):,.1f} km high.")
show(SEL.drop(columns=["geometry", "_k"]).head(12).set_index("wetland_id"))
fig, ax = plt.subplots(figsize=(7, 5.5)); SEL.plot(ax=ax, color="#76d7c4", edgecolor="k", linewidth=0.5)
for r in SEL.itertuples():
    if len(SEL) <= 40: ax.text(r.geometry.representative_point().x, r.geometry.representative_point().y, str(r.wetland_id), fontsize=7, ha="center", color="#c0392b", fontweight="bold")
ax.set_aspect(1 / np.cos(np.radians(float(cen.y.mean())))); ax.set_title(f"{len(SEL)} chosen wetland(s)"); plt.show()
SEL.drop(columns="_k").to_file(OUT / "selected_wetlands.gpkg", layer="selected_wetlands", driver="GPKG")
print("The chosen wetlands are saved in", OUT / "selected_wetlands.gpkg")

## Section B. The chosen wetlands through time

**The question.** For the wetlands you chose, how have their wetness and their cover changed, and does rainfall explain it?

For every chosen wetland you get: the wet area of each year, its trend, the link to rainfall, the **wetness-history** classes inside it, and its cover types by year. The results of all wetlands are in tables with a `wetland_id` column, so one table serves all of them.

**If Section A already worked out the wetness history** with the same satellite, index, months and years, it is **shown for your wetlands and not calculated again**: the yearly water images are reused too. If you change any of those settings, the data are downloaded again for the box around your wetlands.

Section B has two steps.

1. **Look at the rainfall.** Choose a rainfall dataset and the years. The notebook shows the typical rainfall in each month and suggests a rainy season: the five wettest months that fall inside one calendar year. One regional rainfall series is used for all your wetlands, unless they are more than 1000 km apart.
   - *ERA5-Land, monthly*: 11 km cells, 1950 to now, the default.
   - *ERA5-Land, daily*: the same cells, added up month by month; it is slower.
   - *CHIRPS, daily*: 5 km cells, 1981 to now, between 50 S and 50 N; it is built for rainfall in the tropics and subtropics.
2. **Choose the data and the months, then run.** Choose the satellite, the water index, the pixel size and the years as in Section A. Choose two sets of months, each with a drop-down (*all year*, *suggested rainy season* or *my own range of months*, with the first and last month next to it): the months for the **wetness history** (default *all year*) and the months for the **cover types** (default *suggested rainy season*). Then choose how to export: one set of files for all wetlands, or separate GeoTIFFs and plots for each wetland.

A season that crosses New Year (for example November to March) cannot be treated as one season, because yearly images are built per calendar year. The notebook warns you if your range crosses it.

In [ ]:
#@title Section B, step 1. Look at the rainfall { display-mode: "form" }
RAIN_DATASET = "ERA5-Land, monthly (11 km, 1950 on)"  #@param ["ERA5-Land, monthly (11 km, 1950 on)", "ERA5-Land, daily (11 km, 1950 on)", "CHIRPS, daily (5 km, 1981 on, 50 S to 50 N)"]
RAIN_FIRST_YEAR = 1990  #@param {type:"integer"}
RAIN_LAST_YEAR = 2025  #@param {type:"integer"}

import calendar
def plot_rain_months(clim, cover_months=None, dyn_months=None):
    # Typical rainfall by month. Orange bars: the months used for the cover types. A dark outline: the months used for the wetness history.
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.bar(clim.index, clim.values, color=["#e67e22" if (cover_months and m in cover_months) else "#1a5276" for m in clim.index],
           edgecolor=["k" if (dyn_months and m in dyn_months) else "none" for m in clim.index], linewidth=1.5)
    ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND")); ax.set_ylabel("mm per month")
    ax.set_title("Typical monthly rainfall" + (" (orange: cover-type months" + ("; outlined: wetness-history months)" if dyn_months else ")") if cover_months else ""), fontsize=10)
    plt.show()

assert "SEL" in globals(), "Run the cell 'Choose your wetlands' first."
rain_sets, rain_group = rain_for_selection(SEL, RAIN_FIRST_YEAR, RAIN_LAST_YEAR, RAIN_DATASET)
suggest_by_group = {}
for g, rain in rain_sets.items():
    inside, wrap, clim = suggest_months(rain); suggest_by_group[g] = (inside, clim, wrap)
inside0, clim0, wrap0 = suggest_by_group[0]
if len(rain_sets) > 1: print(f"Your wetlands form {len(rain_sets)} groups more than 1000 km apart. The chart below is for the first group; the others are in the saved rainfall files.")
plot_rain_months(clim0)
print("Wettest months on average:", ", ".join(f"{calendar.month_abbr[m]} {v:.0f} mm" for m, v in clim0.sort_values(ascending=False).head(5).items()))
print(f"Suggested rainy season, the five wettest months inside one calendar year: {inside0} ({', '.join(calendar.month_abbr[m] for m in inside0)})")
if wrap0[0] > wrap0[-1]:
    print(f"Note: the five wettest months run across New Year ({wrap0}). A yearly image cannot use them, so the suggestion above is the best inside one calendar year.")
print(f"Rainfall is {RAIN_DATASET.split(',')[0]}: it describes the region around the wetlands, not each wetland. Next: step 2.")

In [ ]:
#@title Section B, step 2. Choose the data and the months, then run { display-mode: "form" }
SENSOR_B = "Landsat, all missions (1984 on)"  #@param ["Landsat, all missions (1984 on)", "Landsat 5 (1984 to 2012)", "Landsat 7 (1999 on, stripes after 2003)", "Landsat 8 (2013 on)", "Landsat 9 (2021 on)", "Sentinel-2 (2015 on)", "MODIS Terra (2000 on, 500 m)", "MODIS Aqua (2002 on, 500 m)", "MODIS Terra and Aqua (2002 on, 500 m)"]
WATER_INDEX_B = "AWEIsh"  #@param ["AWEIsh", "AWEInsh", "MNDWI", "NDWI"]
FIRST_YEAR_B = 1990  #@param {type:"integer"}
LAST_YEAR_B = 2025  #@param {type:"integer"}
PIXEL_SIZE_B = "automatic"  #@param ["automatic", "10", "20", "30", "60", "90", "250", "500", "1000"]
DYNAMICS_MONTHS_MODE = "all year"  #@param ["all year", "suggested rainy season", "my own range of months"]
DYNAMICS_FIRST_MONTH = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
DYNAMICS_LAST_MONTH = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
COVER_TYPE_MONTHS_MODE = "suggested rainy season"  #@param ["all year", "suggested rainy season", "my own range of months"]
COVER_TYPE_FIRST_MONTH = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
COVER_TYPE_LAST_MONTH = "December"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
COVER_TYPES_BY_YEAR = True  #@param {type:"boolean"}
MINIMUM_WET_YEARS_PERCENT = 25  #@param {type:"slider", min:5, max:60, step:5}
PERSISTENT_YEARS_PERCENT = 75  #@param {type:"slider", min:50, max:100, step:5}
YEARS_PER_PERIOD = 3  #@param {type:"slider", min:1, max:10, step:1}
MAX_SCENE_CLOUD_PERCENT_B = 40  #@param {type:"slider", min:10, max:80, step:5}
REUSE_SECTION_A = True  #@param {type:"boolean"}
RUN_EVEN_IF_LARGE = False  #@param {type:"boolean"}
EXPORT_LAYOUT_B = "one set of files for all wetlands"  #@param ["one set of files for all wetlands", "separate GeoTIFF and plots for each wetland"]
RUN_NAME_B = ""  #@param {type:"string"}
# RUN_NAME_B: empty writes to outputs/<project>/B_wetlands_through_time/. A name (for example landsat_all_year) writes to a subfolder, so that runs with different settings do not overwrite each other.
# SENSOR_B, WATER_INDEX_B, PIXEL_SIZE_B, the years: as in Section A.
# *_MONTHS_MODE: all year, suggested rainy season (from step 1), or my own range of months (the first and last month next to it, both included).
# DYNAMICS_*: the months of each year behind the wetness history and the wet-area chart. COVER_TYPE_*: the months behind the cover types.
# COVER_TYPES_BY_YEAR: untick to skip the cover types by year (it needs a second download).
# The three sliders are the dynamics settings, the same as in Section A.
# RUN_EVEN_IF_LARGE: tick to try a request the notebook thinks is too large (it may be slow or fail).
# REUSE_SECTION_A: when Section A used the same sensor, index, months and years, reuse its images and wetness history (no new calculation).
# EXPORT_LAYOUT_B: one set of files for all wetlands (tables have a wetland_id column), or in addition a separate GeoTIFF and plot for each wetland.

assert "SEL" in globals() and "rain_sets" in globals(), "Run 'Choose your wetlands' and step 1 (rainfall) first."
sensor_b, scale_b, y0b, y1b = sensor_settings(SENSOR_B, FIRST_YEAR_B, LAST_YEAR_B, PIXEL_SIZE_B)
suggest_b = suggest_by_group[0][0]
dyn_months = choose_months(DYNAMICS_MONTHS_MODE, DYNAMICS_FIRST_MONTH, DYNAMICS_LAST_MONTH, suggest_b)              # None means all year
months_b = choose_months(COVER_TYPE_MONTHS_MODE, COVER_TYPE_FIRST_MONTH, COVER_TYPE_LAST_MONTH, suggest_b) or list(range(1, 13))
for lab_, mm in (("wetness-history", dyn_months), ("cover-type", months_b)):
    if mm and mm != list(range(min(mm), max(mm) + 1)):
        print(f"Note: the {lab_} months cross New Year. Each yearly image takes them from one calendar year, so it mixes the end of one season with the start of the next. Prefer months inside one calendar year.")
plot_rain_months(suggest_by_group[0][1], months_b, dyn_months)
separate = EXPORT_LAYOUT_B.startswith("separate")
region = region_box(SEL); rb = region.bounds; cloud_b = MAX_SCENE_CLOUD_PERCENT_B
nx_b, ny_b = check_box(rb, scale_b, "box around your wetlands")

# ---- the water images and the wetness history: reused from Section A when they match, otherwise downloaded
def covers(a, b, tol=2 * scale_b / 111319.49): return a[0] - tol <= b[0] and a[1] - tol <= b[1] and a[2] + tol >= b[2] and a[3] + tol >= b[3]       # tolerance of two pixels
have_a = "STATE_A" in globals()
diffs = []
if have_a:
    for what_, a_, b_ in (("satellite", STATE_A["sensor"], sensor_b), ("pixel size", STATE_A["scale"], scale_b), ("water index", STATE_A["index"], WATER_INDEX_B), ("months", STATE_A["months"], dyn_months),
                          ("years", (STATE_A["y0"], STATE_A["y1"]), (y0b, y1b))):
        if a_ != b_: diffs.append(what_)
    if not covers(STATE_A["bounds"], bounds_of(SEL, 0.0)): diffs.append("area (some wetlands lie outside the Section A area)")
same = bool(REUSE_SECTION_A and have_a and not diffs)
if same:
    ra = STATE_A["bounds"]; rb_a = (max(rb[0], ra[0]), max(rb[1], ra[1]), min(rb[2], ra[2]), min(rb[3], ra[3]))
    aw_b = STATE_A["aw"].rio.clip_box(*rb_a)
    print("Section A already has the yearly water images for this satellite, index, months and years: they are reused, with no new download.")
else:
    if REUSE_SECTION_A and have_a:
        print("Section A cannot be reused because these differ from the choices above: " + ", ".join(diffs) + ". Downloading for the box around your wetlands.")
    elif REUSE_SECTION_A: print("Section A was not run in this session, so the data are downloaded.")
    check_workload(y1b - y0b + 1, nx_b, ny_b, "yearly water images", RUN_EVEN_IF_LARGE, bands=1)
    print(f"Downloading {y1b - y0b + 1} yearly images of {WATER_INDEX_B} from {SENSOR_B}, pixel {scale_b} m (a few minutes)...")
    aw_b = water_images(region.__geo_interface__, y0b, y1b, sensor_b, scale_b, cloud_b, WATER_INDEX_B, dyn_months)
check_dynamics_settings(aw_b.sizes["time"], YEARS_PER_PERIOD, MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT)
ok_b = aw_b.notnull().any("time")
if same and (STATE_A["nYear"], STATE_A["thr_wet"], STATE_A["thr_persis"]) == (YEARS_PER_PERIOD, MINIMUM_WET_YEARS_PERCENT, PERSISTENT_YEARS_PERCENT):
    dyn_b = STATE_A["dyn"].rio.clip_box(*rb_a)
    print("Wetness history: taken from Section A (same settings), not calculated again.")
else:
    dyn_b = classify_dynamics(aw_b, nYear=YEARS_PER_PERIOD, thresholdWet=MINIMUM_WET_YEARS_PERCENT, thresholdPersis=PERSISTENT_YEARS_PERCENT, nan_policy="valid").where(ok_b)
labels_b = wetland_labels(SEL, aw_b); lab = labels_b.values; ks = SEL["_k"].values; ids = SEL["wetland_id"].values
px_ha = pixel_area_m2(aw_b) / 1e4; npx = ndimage.sum(np.ones(lab.shape), lab, ks)
out_b = OUT / "B_wetlands_through_time" / re.sub(r"[^A-Za-z0-9_.-]+", "_", RUN_NAME_B); out_b.mkdir(parents=True, exist_ok=True)
if any(out_b.iterdir()): print(f"Note: {out_b} already holds results of an earlier run; files with the same names are overwritten. Use RUN_NAME_B to keep runs apart.")
if float(npx.min()) < 4: print(f"Note: {int((npx < 4).sum())} wetland(s) cover fewer than 4 pixels at {scale_b} m, so their numbers are rough. A finer pixel size or Sentinel-2 would help.")

# ---- wetness history inside each wetland (from the classes above)
codes = list(DYNAMICS_CLASSES)
tab, nvalid = class_counts(dyn_b.values, lab, ks, codes)
hist = pd.DataFrame(tab * px_ha, columns=[DYNAMICS_CLASSES[c] + " (ha)" for c in codes]); hist.insert(0, "wetland_id", ids)
hist.insert(1, "outline_ha", SEL["area_ha"].values); hist.insert(2, "analysed_ha", nvalid * px_ha)
print("Wetness history inside the wetlands (area of each class, in hectares):"); show(hist.head(12).round(2).set_index("wetland_id"))

# ---- wet area by year in each wetland
wet = (aw_b > 0).where(aw_b.notnull()); T = aw_b.sizes["time"]; years = np.asarray(pd.DatetimeIndex(aw_b.time.values).year)
wet_ha = np.full((T, len(ks)), np.nan)
for t in range(T):
    a = wet.isel(time=t).values; cnt = ndimage.sum(np.nan_to_num(a), lab, ks); nv = ndimage.sum(np.isfinite(a).astype(float), lab, ks)
    wet_ha[t] = np.where(nv >= 0.8 * np.maximum(npx, 1), cnt * px_ha, np.nan)                      # a year needs at least 80% of the wetland's pixels
wet_df = pd.DataFrame(wet_ha, index=years, columns=ids); wet_df.index.name = "year"
gaps_b = [int(y) for y in wet_df.index[wet_df.isna().all(axis=1)]]
total_ts = wet_df.sum(axis=1, skipna=False)

# ---- rainfall in the 12 months up to the last month of the yearly image, per rainfall group
end_m = max(dyn_months) if dyn_months else 12
rain_name = f"rain in the 12 months up to the end of {MONTH_NAMES[end_m - 1]} (mm)"
def rain_12(df, years_):
    ep = df.set_index("date")["precip_mm"]
    return pd.Series({int(y): ep[pd.Timestamp(int(y) - 1, end_m, 1) + pd.offsets.MonthBegin(1): pd.Timestamp(int(y), end_m, 1)].sum() for y in years_})
rain_y = {g: rain_12(df, wet_df.index) for g, df in rain_sets.items()}
for g, df in rain_sets.items(): df.to_csv(out_b / f"rainfall_monthly_group{g}.csv", index=False)

rows = []
for j, wid in enumerate(ids):
    s = wet_df[wid].dropna(); r = dict(wetland_id=wid, outline_ha=SEL["area_ha"].iloc[j], years_with_data=len(s))
    if len(s) >= 4:
        lr = linregress(s.index.values.astype(float), s.values); k5 = min(5, max(len(s) // 3, 1))
        r.update(mean_wet_ha=s.mean(), min_wet_ha=s.min(), min_year=int(s.idxmin()), max_wet_ha=s.max(), max_year=int(s.idxmax()), first_years_mean_ha=s.iloc[:k5].mean(), last_years_mean_ha=s.iloc[-k5:].mean(),
                 trend_ha_per_decade=lr.slope * 10, trend_p=lr.pvalue)
        ry = rain_y[int(rain_group.loc[wid])].reindex(s.index)
        if len(s) >= 8 and ry.notna().all(): r["rain_correlation"], r["rain_correlation_p"] = spearmanr(ry.values, s.values)
    rows.append(r)
summary = pd.DataFrame(rows)
summary = summary.merge(hist.drop(columns=["outline_ha"]), on="wetland_id")
lines = [f"{len(SEL)} wetland(s), {SEL['area_ha'].sum():,.1f} ha in total. Satellite: {SENSOR_B}, water index {WATER_INDEX_B}, pixel {scale_b} m, years {y0b} to {y1b}.",
         f"Wetness history built from the 90th percentile of the water index over {'all months' if dyn_months is None else f'months {dyn_months}'} of each year; dynamics settings: wetland if wet in at least {MINIMUM_WET_YEARS_PERCENT}% of years, persistent if wet in at least {PERSISTENT_YEARS_PERCENT}%, early and recent periods of {YEARS_PER_PERIOD} years."]
if "trend_p" in summary:
    ok_t = summary.dropna(subset=["trend_p"]); sig = ok_t[ok_t["trend_p"] < 0.05]
    lines.append(f"Trend in wet area: {int((sig['trend_ha_per_decade'] > 0).sum())} wetland(s) clearly increasing and {int((sig['trend_ha_per_decade'] < 0).sum())} clearly decreasing (p < 0.05); no clear trend in {len(ok_t) - len(sig)}. Read trends as directions over noisy yearly values, not precise rates.")
if "rain_correlation" in summary:
    lines.append(f"Wet area and {rain_name}: median correlation {summary['rain_correlation'].median():+.2f} across wetlands (1 means they rise and fall together, 0 means no link).")
if gaps_b: lines.append(f"Years with no usable data, left out of the statistics: {', '.join(map(str, gaps_b))}.")
print("\n".join("- " + l for l in lines)); print("\nWet area by year, trend and rainfall link for each wetland (the first 12):")
show(summary[["wetland_id", "outline_ha", "years_with_data", "mean_wet_ha", "trend_ha_per_decade", "trend_p", "rain_correlation"]].head(12).round(2).set_index("wetland_id"))

# ---- figures: the wetness-history map, the total wet area by year, and the largest wetlands one by one
fig, ax = plot_dynamics(dyn_b, title=f"Wetness history, {len(SEL)} chosen wetland(s) (outlined)", figsize=(10, 5.4)); plot_outlines(ax, SEL)
ax.set_aspect(1 / np.cos(np.radians(float(dyn_b.y.mean())))); fig.savefig(out_b / "wetness_history_map.png", dpi=150, bbox_inches="tight"); plt.show()
fig, ax = plt.subplots(figsize=(11, 3.8)); ax.bar(total_ts.index, total_ts.values, color="#1a5276")
for y in gaps_b: ax.axvspan(y - 0.5, y + 0.5, color="#e5e8ea", zorder=0)
ax.axhline(SEL["area_ha"].sum(), color="#c0392b", ls="--", lw=1, label="outlines"); ax.legend(loc="upper left", fontsize=8); ax.set_ylabel("wet area (ha)")
ax.set_title(f"Wet area by year, total of {len(SEL)} wetland(s)" + (" (grey = no usable data)" if gaps_b else ""))
if len(rain_sets) == 1:
    ax2 = ax.twinx(); ax2.spines["right"].set_visible(True); r_ = rain_y[0].reindex(total_ts.index); ax2.plot(r_.index, r_.values, color="#e67e22", marker="o", ms=3); ax2.set_ylabel(rain_name, color="#e67e22", fontsize=8)
fig.tight_layout(); fig.savefig(out_b / "wet_area_by_year_total.png", dpi=150, bbox_inches="tight"); plt.show()
top = SEL.sort_values("area_ha", ascending=False).head(12)["wetland_id"].tolist()
if len(SEL) > 1:
    nrow = int(np.ceil(len(top) / 3)); fig, axs = plt.subplots(nrow, 3, figsize=(13, 2.6 * nrow), squeeze=False, sharex=True)
    for ax, wid in zip(axs.ravel(), top): ax.bar(wet_df.index, wet_df[wid].values, color="#1a5276"); ax.set_title(f"wetland {wid}", fontsize=9)
    for ax in axs.ravel()[len(top):]: ax.axis("off")
    fig.suptitle("Wet area by year (ha), the largest wetlands" if len(SEL) > len(top) else "Wet area by year (ha)", fontsize=10); fig.tight_layout(); fig.savefig(out_b / "wet_area_by_year_each_wetland.png", dpi=150, bbox_inches="tight"); plt.show()

# ---- cover types by year
cover_long = None
if COVER_TYPES_BY_YEAR:
    check_workload(y1b - y0b + 1, nx_b, ny_b, "yearly cover-type images", RUN_EVEN_IF_LARGE, bands=3)
    print(f"Downloading cover-type images (MNDWI, NDVI, NDTI) for {y1b - y0b + 1} years, months {months_b} (a few minutes)...")
    ct = cover_indices(region.__geo_interface__, y0b, y1b, sensor_b, scale_b, cloud_b, months_b)
    valid_c = ct.to_array().notnull().all("variable")
    wct_cls = classify_wct_ema(ct)["wetland_cover_type"].where(valid_c)
    lab_c_da = wetland_labels(SEL, ct["MNDWI"]); lab_c = lab_c_da.values; px_c = pixel_area_m2(ct["MNDWI"]) / 1e4; npx_c = ndimage.sum(np.ones(lab_c.shape), lab_c, ks)
    yrs_c = np.asarray(pd.DatetimeIndex(ct.time.values).year); recs = []
    for t, y in enumerate(yrs_c):
        tb, nv = class_counts(wct_cls.isel(time=t).values, lab_c, ks, list(WCT_CLASSES))
        good = nv >= 0.8 * np.maximum(npx_c, 1)
        for j, wid in enumerate(ids):
            if good[j]: recs += [dict(wetland_id=wid, year=int(y), class_code=c, cover_type=WCT_CLASSES[c], area_ha=tb[j, i] * px_c) for i, c in enumerate(WCT_CLASSES) if c > 0]
    cover_long = pd.DataFrame(recs)
    if len(cover_long):
        tot = cover_long.groupby(["year", "cover_type"])["area_ha"].sum().unstack().reindex(columns=[WCT_CLASSES[c] for c in range(1, 6)]).reindex(range(int(yrs_c.min()), int(yrs_c.max()) + 1)).fillna(0)
        fig, ax = plt.subplots(figsize=(11, 3.8))
        for y_ in tot.index[(tot.sum(axis=1) == 0)]: ax.axvspan(y_ - 0.5, y_ + 0.5, color="#e5e8ea", zorder=0)
        ax.stackplot(tot.index, [tot[WCT_CLASSES[c]] for c in range(1, 6)], labels=[WCT_CLASSES[c] for c in range(1, 6)], colors=[WCT_COLORS[c] for c in range(1, 6)])
        ax.legend(loc="upper left", fontsize=7, ncol=3); ax.set_ylabel("area inside the outlines (ha)"); ax.set_title(f"Cover types by year, total of {len(SEL)} wetland(s) (months {months_b}; grey = no usable data)")
        fig.tight_layout(); fig.savefig(out_b / "cover_types_by_year_total.png", dpi=150, bbox_inches="tight"); plt.show()
        cover_long.round(3).to_csv(out_b / "cover_types_by_year_by_wetland.csv", index=False)
        print("Note: 'Emergent / Floating Vegetation' can also be dry land with green plants. Check that water is present before treating it as wetland.")
    else:
        print("No year had enough clear data for the cover types.")

# ---- files
SEL.drop(columns="_k").to_file(out_b / "selected_wetlands.gpkg", layer="selected_wetlands", driver="GPKG")
summary.round(3).to_csv(out_b / "wetland_summary.csv", index=False)
wet_df.round(3).reset_index().melt(id_vars="year", var_name="wetland_id", value_name="wet_area_ha").dropna().to_csv(out_b / "wet_area_by_year_by_wetland.csv", index=False)
hist.round(3).to_csv(out_b / "wetness_history_by_wetland.csv", index=False)
dyn_sel = dyn_b.where(labels_b > 0)
if cover_long is not None and len(cover_long):
    last_y = int(cover_long["year"].max()); last_t = int(np.where(yrs_c == last_y)[0][0])             # the last year with enough data
if not separate:
    write_geotiff(dyn_sel, out_b / "wetness_history_chosen_wetlands.tif", DYNAMICS_COLORS)
    if cover_long is not None and len(cover_long): write_geotiff(wct_cls.isel(time=last_t).where(lab_c_da > 0), out_b / f"cover_types_{last_y}_chosen_wetlands.tif", WCT_COLORS)
else:
    n_sep = min(len(SEL), 100)
    if len(SEL) > 100: print("Separate files are made for the 100 largest wetlands only; the tables hold all of them.")
    for _, r in SEL.sort_values("area_ha", ascending=False).head(n_sep).iterrows():
        wid, k, geom = r["wetland_id"], int(r["_k"]), r["geometry"]
        write_geotiff(crop_to(dyn_b.where(labels_b == k), geom), out_b / f"wetland_{wid}_wetness_history.tif", DYNAMICS_COLORS)
        if cover_long is not None and len(cover_long): write_geotiff(crop_to(wct_cls.isel(time=last_t).where(lab_c_da == k), geom), out_b / f"wetland_{wid}_cover_types_{last_y}.tif", WCT_COLORS)
        fig, axs = plt.subplots(1, 2, figsize=(11, 3.6)); plot_dynamics(crop_to(dyn_b, geom), ax=axs[0], title=f"wetland {wid}: wetness history", legend_loc="lower right")
        gpd.GeoSeries([geom], crs=4326).boundary.plot(ax=axs[0], color="k", linewidth=1.2); axs[0].set_aspect(1 / np.cos(np.radians(float(dyn_b.y.mean()))))
        axs[1].bar(wet_df.index, wet_df[wid].values, color="#1a5276"); axs[1].set_ylabel("wet area (ha)"); axs[1].set_title(f"wetland {wid}: wet area by year")
        fig.tight_layout(); fig.savefig(out_b / f"wetland_{wid}_wetness_and_wet_area.png", dpi=130, bbox_inches="tight"); plt.close(fig)
(out_b / "report.md").write_text("# Wetlands through time\n\n" + "\n".join("- " + l for l in lines) + "\n")
build_run_manifest(parameters={"section": "B", "wetlands": [str(i) for i in ids[:200]], "sensor": SENSOR_B, "water_index": WATER_INDEX_B, "pixel_m": scale_b, "years": [y0b, y1b], "nYear": YEARS_PER_PERIOD,
                               "thresholdWet": MINIMUM_WET_YEARS_PERCENT, "thresholdPersis": PERSISTENT_YEARS_PERCENT, "wetness_history_months": dyn_months or "all year", "cover_type_months": months_b,
                               "rainfall": RAIN_DATASET, "section_A_reused": bool(same), "export_layout": EXPORT_LAYOUT_B, "max_cloud_percent": cloud_b}, output_path=out_b / "run_settings.json")
print("\nFiles are in", out_b, "\nNEXT: save these results (the next cell).")

**How to read it**

- **The tables** all have a `wetland_id` column. `wetland_summary.csv` has one row per wetland: the wet area by year (mean, lowest, highest), its trend per decade, the rainfall correlation and the wetness-history classes. The long tables (`wet_area_by_year_by_wetland.csv`, `cover_types_by_year_by_wetland.csv`) hold one row per wetland and year.
- **The wetness-history map** colours the wetland pixels by class (see Section A) for the region around your wetlands; your wetlands are outlined and numbered.
- **Wet area by year** is the area of pixels whose water index is above zero. A year counts only if at least 80% of the wetland's pixels have data. Grey years had no usable data: they are left out, not counted as dry.
- **The trend** is a direction over noisy yearly values, not a precise rate. A wetland driven by rainfall can show a trend only because the record starts in a wet or dry run. **The correlation** with rainfall: 1 means they rise and fall together, 0 means no link.
- **Rainfall is regional.** One series for the whole box, unless your wetlands are more than 1000 km apart. It is good for timing the rainy season, less good for explaining one small wetland.
- **Cover types by year** show whether open water, vegetation or moist soil change with wetness. **Emergent / floating vegetation can be dry land**: green plants look the same over a wet reed bed and over grass or a crop. Check that water is present before you treat that class as wetland.
- **Small wetlands and large pixels.** A wetland of fewer than four pixels has rough numbers. MODIS (500 m) suits large lakes and lagoons only.
- **If Section A was reused**, the wetness history is the one from Section A (the run settings file says so). Settings you change here are applied to the data again.

Files are in `outputs/<project>/B_wetlands_through_time/`: the chosen wetlands (GeoPackage), the tables, the maps and charts, the GeoTIFFs (one for all wetlands, or one for each if you chose separate files), the rainfall series, a report and a record of your settings.

In [ ]:
#@title Save the results of Section B { display-mode: "form" }
SAVE_TO = "zip in the outputs folder"  #@param ["zip in the outputs folder", "download zip (Colab)", "copy to Google Drive (Colab)", "copy to a folder"]
DESTINATION = ""  #@param {type:"string"}
# download zip (Colab): the browser downloads a zip of this section's folder.
# copy to Google Drive (Colab): DESTINATION is a folder name inside My Drive (default WetlandMapper_results).
# copy to a folder: DESTINATION is a folder path on your computer, for example ~/wetland_results.
save_step("B_wetlands_through_time", SAVE_TO, DESTINATION)

## Section C. Cover types for a period

**The question.** What cover types does the satellite see in your chosen wetlands in the period you choose?

You choose the **satellite**, the **period** (a start year and month, an end year and month, or simply the last N days) and **how to composite** it:

- **one composite for the whole period**: every scene in the period is downloaded and the median is taken. Best for a season or a few months.
- **one composite per year**: the same months of every year in the period, one image per year. Good for a time series of the same season.
- **one composite per month**: one image for each month of the period. This is **heavy**: many images from a large area are held in memory and can fail or take very long in Earth Engine. The notebook estimates the size, warns you and stops, unless you tick *RUN_EVEN_IF_LARGE* to try anyway.

You get **Level 1** (five cover types) and **Level 2** (twelve more detailed cover types, plus non-wetland) or either of them. Level 2 separates deep and shallow clear water, highly and moderately turbid water, moist soil, submerged and emergent vegetation, mixed classes and the fringes between them.

The three indices behind the classes are MNDWI (open water), NDVI (green vegetation) and NDTI (turbidity), as in notebook 1.

**Cautions.**

- **Emergent / floating vegetation can be dry land with green plants** (grass, crops, shrubs). Check that water is present before you treat that class as wetland.
- In **tidal places** the image shows the tide at the moment of the satellite pass.
- The classes describe what the satellite signal looks like, not a verified habitat. Your field notes are the check.
- Do not mix sensors in one time series: the same wetland can change class when the sensor changes. Keep one sensor for one series.

In [ ]:
#@title Section C. Cover types for a period { display-mode: "form" }
SENSOR_C = "Sentinel-2 (2015 on)"  #@param ["Landsat, all missions (1984 on)", "Landsat 5 (1984 to 2012)", "Landsat 7 (1999 on, stripes after 2003)", "Landsat 8 (2013 on)", "Landsat 9 (2021 on)", "Sentinel-2 (2015 on)", "MODIS Terra (2000 on, 500 m)", "MODIS Aqua (2002 on, 500 m)", "MODIS Terra and Aqua (2002 on, 500 m)"]
PERIOD = "my own period"  #@param ["my own period", "the last N days"]
START_YEAR = 2024  #@param {type:"integer"}
START_MONTH = "January"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
END_YEAR = 2024  #@param {type:"integer"}
END_MONTH = "March"  #@param ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
DAYS_BACK = 20  #@param {type:"slider", min:5, max:120, step:5}
COMPOSITE_MODE = "one composite for the whole period"  #@param ["one composite for the whole period", "one composite per year (same months each year)", "one composite per month (heavy)"]
WCT_LEVELS = "level 1 and level 2"  #@param ["level 1 and level 2", "level 1 only", "level 2 only"]
PIXEL_SIZE_C = "automatic"  #@param ["automatic", "10", "20", "30", "60", "90", "250", "500", "1000"]
MAX_SCENE_CLOUD_PERCENT_C = 60  #@param {type:"slider", min:10, max:90, step:5}
MINIMUM_VALID_PERCENT = 40  #@param {type:"slider", min:10, max:90, step:5}
RUN_EVEN_IF_LARGE = False  #@param {type:"boolean"}
MASK_TO_WETLANDS = True  #@param {type:"boolean"}
EXPORT_LAYOUT_C = "one set of files for all wetlands"  #@param ["one set of files for all wetlands", "separate GeoTIFF and plots for each wetland"]
FIELD_POINTS = "none"  #@param ["none", "upload", "path in Drive or Colab"]
FIELD_POINTS_PATH = ""  #@param {type:"string"}
RUN_NAME_C = ""  #@param {type:"string"}
# RUN_NAME_C: empty writes to outputs/<project>/C_cover_types/. A name (for example s2_jan_mar_2024) writes to a subfolder, so that runs with different settings do not overwrite each other.
# PERIOD: your own start and end month, or the last DAYS_BACK days up to today. The end month is included.
# COMPOSITE_MODE: see the text above. 'per year' uses the months from START_MONTH to END_MONTH in every year from START_YEAR to END_YEAR.
# WCT_LEVELS: Level 1 (five cover types), Level 2 (more detailed), or both.
# MINIMUM_VALID_PERCENT: a scene or composite with less clear data than this inside your wetlands is left out.
# RUN_EVEN_IF_LARGE: tick to try a request the notebook thinks is too large. It may fail or be slow in Earth Engine.
# MASK_TO_WETLANDS: write the GeoTIFFs for your wetlands only (pixels outside them are empty); untick to keep the whole box.
# EXPORT_LAYOUT_C: one set of files for all wetlands (tables have a wetland_id column), or in addition a GeoTIFF and a plot for each wetland.
# FIELD_POINTS: optional points (CSV with lat and lon columns, or KMZ, KML or GeoPackage of points) to read the cover type at each point.

assert "SEL" in globals(), "Run the cell 'Choose your wetlands' first."
today = dt.date.today(); mode = {"one composite for the whole period": "one", "one composite per year (same months each year)": "year", "one composite per month (heavy)": "month"}[COMPOSITE_MODE]
if PERIOD == "the last N days":
    start_d, end_d = today - dt.timedelta(days=int(DAYS_BACK)), today
    if mode != "one": print("For the last N days one composite is made."); mode = "one"
else:
    sm, em = MONTH_NAMES.index(START_MONTH) + 1, MONTH_NAMES.index(END_MONTH) + 1
    start_d = dt.date(int(START_YEAR), sm, 1)
    end_d = min((dt.date(int(END_YEAR), em + 1, 1) if em < 12 else dt.date(int(END_YEAR) + 1, 1, 1)) - dt.timedelta(days=1), today)
sensor_c, scale_c, y0c, y1c = sensor_settings(SENSOR_C, start_d.year, end_d.year, PIXEL_SIZE_C)
start_d, end_d = max(start_d, dt.date(y0c, 1, 1)), min(end_d, dt.date(y1c, 12, 31))
if start_d > end_d: raise ValueError("The start is after the end. Check the period.")
months_c = choose_months("my own range of months", START_MONTH, END_MONTH) if (mode == "year" and PERIOD != "the last N days") else None
if months_c and months_c != list(range(min(months_c), max(months_c) + 1)):
    print("Note: the months cross New Year. Each yearly composite takes them from one calendar year, so it mixes two seasons.")
region = region_box(SEL); rb = region.bounds
nx, ny = check_box(rb, scale_c, "box around your wetlands")
n_months = (end_d.year - start_d.year) * 12 + end_d.month - start_d.month + 1
n_img = {"one": max(1, (end_d - start_d).days // SENSORS[SENSOR_C]["revisit"]), "year": end_d.year - start_d.year + 1, "month": n_months}[mode]
check_workload(n_img, nx, ny, f"{COMPOSITE_MODE} ({start_d} to {end_d})", RUN_EVEN_IF_LARGE, bands=3)
print(f"Downloading about {n_img} image(s) of MNDWI, NDVI and NDTI from {SENSOR_C}, pixel {scale_c} m, {start_d} to {end_d} ({nx} x {ny} pixels). This can take several minutes...")
ds = period_indices(region.__geo_interface__, start_d, end_d, sensor_c, scale_c, MAX_SCENE_CLOUD_PERCENT_C, mode, months_c)

lab_da = wetland_labels(SEL, ds["MNDWI"]); lab = lab_da.values; ks = SEL["_k"].values; ids = SEL["wetland_id"].values
px_ha = pixel_area_m2(ds["MNDWI"]) / 1e4; inside = lab_da > 0
valid = ds.to_array().notnull().all("variable"); vpct = (valid.where(inside).mean(("y", "x")) * 100).values
t_idx = pd.DatetimeIndex(ds.time.values)
out_c = OUT / "C_cover_types" / re.sub(r"[^A-Za-z0-9_.-]+", "_", RUN_NAME_C); out_c.mkdir(parents=True, exist_ok=True)
if any(out_c.iterdir()): print(f"Note: {out_c} already holds results of an earlier run; files with the same names are overwritten. Use RUN_NAME_C to keep runs apart.")
if mode == "one":
    scenes = pd.DataFrame({"time (UTC)": t_idx.strftime("%Y-%m-%d %H:%M"), "% clear inside the wetlands": vpct.round(0), "used": vpct >= MINIMUM_VALID_PERCENT})
    print(f"{int(scenes['used'].sum())} of {len(scenes)} scenes have at least {MINIMUM_VALID_PERCENT}% clear data inside your wetlands and are used."); show(scenes.head(25))
    scenes.to_csv(out_c / "scenes.csv", index=False)
    if not scenes["used"].any(): raise ValueError("No scene is clear enough. Lower MINIMUM_VALID_PERCENT, use a longer period, or allow more cloud.")
    keep = np.where(vpct >= MINIMUM_VALID_PERCENT)[0]; comp = ds.isel(time=keep).median("time", skipna=True)
    mid = t_idx[keep][len(keep) // 2]; ds = comp.expand_dims(time=[np.datetime64(mid)]); labels_t = [f"{start_d:%Y-%m-%d}_to_{end_d:%Y-%m-%d}"]
    nice = [f"{start_d:%d %b %Y} to {end_d:%d %b %Y}"]
else:
    good = vpct >= MINIMUM_VALID_PERCENT
    if not good.all(): print(f"{int((~good).sum())} composite(s) have less than {MINIMUM_VALID_PERCENT}% clear data inside your wetlands and are left out.")
    if not good.any(): raise ValueError("No composite is clear enough. Lower MINIMUM_VALID_PERCENT or choose another period or sensor.")
    ds = ds.isel(time=np.where(good)[0]); t_idx = pd.DatetimeIndex(ds.time.values)
    labels_t = [f"{t:%Y}" if mode == "year" else f"{t:%Y-%m}" for t in t_idx]; nice = labels_t
valid = ds.to_array().notnull().all("variable")

# ---- classify: Level 1 and Level 2. Missing pixels are put back to NaN after classifying.
want1, want2 = WCT_LEVELS != "level 2 only", WCT_LEVELS != "level 1 only"
if want2:
    res = classify_wct_ema_level2(ds); l2 = res["wetland_cover_type_level2"].where(valid); l1 = res["wetland_cover_type"].where(valid)
else:
    l1 = classify_wct_ema(ds)["wetland_cover_type"].where(valid); l2 = None
levels = ([("level1", l1, WCT_CLASSES, WCT_COLORS)] if want1 else []) + ([("level2", l2, WCT_LEVEL2_CLASSES, WCT_LEVEL2_COLORS)] if want2 else [])

# ---- tables: area of each cover type in each wetland, for each composite
npx = ndimage.sum(np.ones(lab.shape), lab, ks)
for name, da, classes, colors in levels:
    recs = []
    for t, lbl in enumerate(labels_t):
        codes = list(classes); tab, nv = class_counts(da.isel(time=t).values, lab, ks, codes)
        for j, wid in enumerate(ids):
            for i, c in enumerate(codes):
                recs.append(dict(wetland_id=wid, composite=lbl, class_code=c, cover_type=classes[c], pixels=int(tab[j, i]), area_ha=round(tab[j, i] * px_ha, 3),
                                 percent_of_clear_area=round(100 * tab[j, i] / nv[j], 1) if nv[j] else np.nan))
    tbl = pd.DataFrame(recs); tbl.to_csv(out_c / f"cover_types_{name}_by_wetland.csv", index=False)
    tot = tbl.groupby(["composite", "cover_type"], sort=False)["area_ha"].sum().unstack().round(2)
    print(f"\n{name.replace('level', 'Level ')}: area of each cover type in all chosen wetlands together (ha):"); show(tot.head(12))
    if len(SEL) > 1 or len(labels_t) > 1: print(f"All {len(SEL)} wetland(s) are in cover_types_{name}_by_wetland.csv (one row per wetland, composite and class).")

# ---- maps (at most 12 composites)
n_show = min(len(labels_t), 12)
if len(labels_t) > n_show: print(f"The maps show the first {n_show} of {len(labels_t)} composites; the GeoTIFFs and tables hold all of them.")
for name, da, classes, colors in levels:
    ncol = min(n_show, 4); nrow = int(np.ceil(n_show / ncol)); fig, axs = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.4 * nrow), squeeze=False)
    for i, ax in enumerate(axs.ravel()):
        if i >= n_show: ax.axis("off"); continue
        (plot_wct if name == "level1" else plot_wct_level2)(da.isel(time=i), ax=ax, title=nice[i], add_colorbar=False)
        SEL.boundary.plot(ax=ax, color="k", linewidth=0.8); ax.set_aspect(1 / np.cos(np.radians(float(ds.y.mean()))))
    class_legend(fig, classes, colors, "Level 1 cover types" if name == "level1" else "Level 2 cover types")
    fig.savefig(out_c / f"cover_types_{name}_maps.png", dpi=140, bbox_inches="tight"); plt.show()

# ---- GeoTIFFs: one per composite and level (for all wetlands), or one per wetland as well
n_files = len(SEL) * len(labels_t) * len(levels); separate = EXPORT_LAYOUT_C.startswith("separate")
if separate and n_files > 300:
    print(f"Separate files would make {n_files} GeoTIFFs. That is too many, so one set of files for all wetlands is written; the tables have a wetland_id column."); separate = False
for name, da, classes, colors in levels:
    for i, lbl in enumerate(labels_t):
        layer = da.isel(time=i); write_geotiff(layer.where(inside) if MASK_TO_WETLANDS else layer, out_c / f"cover_types_{name}_{lbl}.tif", colors)
        if separate:
            for _, r in SEL.iterrows(): write_geotiff(crop_to(layer.where(lab_da == int(r["_k"])), r["geometry"]), out_c / f"wetland_{r['wetland_id']}_cover_types_{name}_{lbl}.tif", colors)
if separate:
    for _, r in SEL.sort_values("area_ha", ascending=False).head(50).iterrows():
        fig, axs = plt.subplots(len(levels), min(len(labels_t), 6), figsize=(3.6 * min(len(labels_t), 6), 3.6 * len(levels)), squeeze=False)
        for row, (name, da, classes, colors) in enumerate(levels):
            for i, ax in enumerate(axs[row]):
                (plot_wct if name == "level1" else plot_wct_level2)(crop_to(da.isel(time=i), r["geometry"]), ax=ax, title=f"{r['wetland_id']}, {nice[i]}", add_colorbar=False)
                gpd.GeoSeries([r["geometry"]], crs=4326).boundary.plot(ax=ax, color="k", linewidth=1)
        class_legend(fig, levels[-1][2], levels[-1][3], "Cover types (" + levels[-1][0].replace("level", "Level ") + ")")
        fig.tight_layout(); fig.savefig(out_c / f"wetland_{r['wetland_id']}_cover_types.png", dpi=120, bbox_inches="tight"); plt.close(fig)

# ---- field points (optional)
if FIELD_POINTS != "none":
    pts = read_points(upload_file() if FIELD_POINTS == "upload" else FIELD_POINTS_PATH)
    px_ = xr.DataArray(pts.geometry.x.values, dims="p"); py_ = xr.DataArray(pts.geometry.y.values, dims="p"); rows = []
    for i, lbl in enumerate(labels_t):
        v = {k: ds[k].isel(time=i).sel(x=px_, y=py_, method="nearest").values for k in ("MNDWI", "NDVI", "NDTI")}
        c1 = l1.isel(time=i).sel(x=px_, y=py_, method="nearest").values if want1 else None; c2 = l2.isel(time=i).sel(x=px_, y=py_, method="nearest").values if want2 else None
        for j, pid in enumerate(pts["point_id"]):
            r = dict(point_id=pid, composite=lbl, MNDWI=round(float(v["MNDWI"][j]), 3), NDVI=round(float(v["NDVI"][j]), 3), NDTI=round(float(v["NDTI"][j]), 3))
            if want1: r["level1"] = WCT_CLASSES.get(int(c1[j]), "no data") if np.isfinite(c1[j]) else "no data"
            if want2: r["level2"] = WCT_LEVEL2_CLASSES.get(int(c2[j]), "no data") if np.isfinite(c2[j]) else "no data"
            rows.append(r)
    ft = pd.DataFrame(rows); ft.to_csv(out_c / "field_points_cover_types.csv", index=False); print("Cover type at your field points:"); show(ft.head(20))
    print("The index values show which way a pixel leans: MNDWI for open water, NDVI for green vegetation, NDTI for turbidity.")

SEL.drop(columns="_k").to_file(out_c / "selected_wetlands.gpkg", layer="selected_wetlands", driver="GPKG")
build_run_manifest(parameters={"section": "C", "wetlands": [str(i) for i in ids[:200]], "sensor": SENSOR_C, "pixel_m": scale_c, "period": [str(start_d), str(end_d)], "composite_mode": COMPOSITE_MODE, "levels": WCT_LEVELS,
                               "months_per_year": months_c, "max_cloud_percent": MAX_SCENE_CLOUD_PERCENT_C, "minimum_valid_percent": MINIMUM_VALID_PERCENT, "export_layout": EXPORT_LAYOUT_C}, output_path=out_c / "run_settings.json")
print("\nFiles are in", out_c, "\nNEXT: save these results (the next cell).")

**How to read it**

- **The scene list** (one-composite mode) shows which scenes went into the composite and how much of your wetlands each scene saw clearly. A scene with little clear data is mostly cloud and is left out. Times are UTC.
- **The tables** give, for each wetland and each composite, the area of every cover type in hectares and as a share of the clear area. `cover_types_level1_by_wetland.csv` and `cover_types_level2_by_wetland.csv` have a `wetland_id` column, one row per wetland, composite and class.
- **The maps** show the first 12 composites. The GeoTIFFs hold all of them, one file per composite and level, with the class colours inside (QGIS shows them without a style file).
- **Level 1 or Level 2?** Level 1 is easier to report. Level 2 separates classes that matter for water quality and vegetation, but each class covers fewer pixels and is more sensitive to the sensor, haze and tide.
- **Emergent / floating vegetation can be dry land.** Check that water is present before you treat that class as wetland.
- **Composites per month and per year** are for time series: compare the same season, with the same sensor.
- If a map looks wrong, check the scene dates first: haze, cloud, a different tide or sun glint on open water are the usual reasons.

In [ ]:
#@title Save the results of Section C { display-mode: "form" }
SAVE_TO = "zip in the outputs folder"  #@param ["zip in the outputs folder", "download zip (Colab)", "copy to Google Drive (Colab)", "copy to a folder"]
DESTINATION = ""  #@param {type:"string"}
# download zip (Colab): the browser downloads a zip of this section's folder.
# copy to Google Drive (Colab): DESTINATION is a folder name inside My Drive (default WetlandMapper_results).
# copy to a folder: DESTINATION is a folder path on your computer, for example ~/wetland_results.
save_step("C_cover_types", SAVE_TO, DESTINATION)

## Reading results, and what this notebook does not do

- Open water is seen well. **Small, vegetated or briefly wet features** (springs, seeps, reed beds) are seen poorly: a 30 m pixel covers 0.09 ha and a MODIS pixel covers 25 ha.
- A yearly image records the wettest cloud-free moment of the months you choose, not the whole year. The choice of months, sensor and index changes the result: report them.
- **Cover-type classes describe the satellite signal**, not a verified habitat. **Emergent / floating vegetation can be dry land.** Open sea, large lakes and reservoirs show as persistent wetland.
- **Do not mix sensors in one time series.** The wetness history and the cover types change when the sensor changes (for example between Landsat and Sentinel-2).
- A wetland on **steep, shaded ground** can be confused with shadow, and in a **dry climate** water can be too brief for a yearly image. WetlandMapper has options for both (`dem_mask=True` and `climate_adaptive=True` in `fetch`), which this notebook does not use. See the WetlandMapper documentation.
- **Always report with the map:** the satellite, the water index, the pixel size, how each yearly image was built (percentile and months), the dynamics settings (wet percentage, persistent percentage, years per period), the years and the software version. The `run_settings.json` files have them.

## If something goes wrong

| Message | What to do |
|---|---|
| "This notebook needs Earth Engine" | Check the project ID in the setup cell and that the Earth Engine API is enabled for your Google Cloud project |
| "is too large for one download" | You are using `fetch`: use xee (*DOWNLOAD_METHOD* in the setup cell), a smaller area, a larger pixel size, or fewer wetlands that lie closer together |
| "This may be slow or fail in Earth Engine" | Many images from a large area, held in memory. Use a shorter period or a larger pixel size, or tick *RUN_EVEN_IF_LARGE* to try anyway |
| "xee or dask is not installed" | Run `pip install xee dask` and restart the notebook, or use `fetch` |
| "These IDs are not in the file" | Check the IDs in the file `*_wetland_patches.csv` from Section A, or the ID column of your file |
| "Run the cell 'Choose your wetlands' first" | Run the selection cell before Section B or C |
| "No scene is clear enough" | Lower the minimum clear percentage, use a longer period, or allow more cloud |
| A year is missing | Earth Engine returned no scenes or a download failed twice. It is left out of the statistics |
| The run is slow | Earth Engine fetches the images one by one: minutes for a small box, much longer for a large one |

### Data and citation

- Software: Singh, M. (2026). WetlandMapper. https://doi.org/10.5281/zenodo.18967176
- Dynamics: Singh, M., and Sinha, R. (2022). *Remote Sensing Letters*, 13(1), 1-13. https://doi.org/10.1080/2150704X.2021.1980919
- Cover types: Singh, M., Allaka, S., Gupta, P. K., Patel, J. G., and Sinha, R. (2022). *Environmental Monitoring and Assessment*, 194(12), 878. https://doi.org/10.1007/s10661-022-10541-7
- Landsat Collection 2 Level-2 (USGS and NASA), Sentinel-2 surface reflectance (ESA and Copernicus), MODIS surface reflectance (NASA), ERA5-Land (Copernicus and ECMWF), CHIRPS (Climate Hazards Center, UC Santa Barbara), through Google Earth Engine (Gorelick et al., 2017, *Remote Sensing of Environment* 202, 18-27).